# Debugging LM Optimizer and KeyFrame Graph

This notebook loads metadata from a run and recreates the **exact optimization scenario** from `modular_pipeline.py`:
- Uses `LocalOptimizer` for per-frame local optimization (as in pipeline)
- Uses `KeyFrameGraph` for global keyframe optimization (as in pipeline)
- Reconstructs `ObjectFrameData` exactly as the pipeline creates it
- Visualizes both local and global optimization results vs GT

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.transform import Rotation as R
from easydict import EasyDict as edict
import gtsam

# Add project root to path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

# Import pipeline components (as used in modular_pipeline.py)
from point2pose.pipeline.components.local_optimizer import LocalOptimizer
from point2pose.pipeline.components.key_frame_graph import KeyFrameGraph
from point2pose.data_types.object_frame_data import ObjectFrameData
from point2pose.data_types.key_frame import KeyFrame
from point2pose.modules.object.object import Object
from point2pose.data_types.point_track_table import PointTrackTable
from point2pose.utils.transform import inverse_SE3, transform_pts

# Import dataset readers for GT pose loading
from point2pose.io.sources.dataset.datareader import Ho3dReader, YcbineoatReader

# Use interactive backend for rotatable 3D plots
# In Jupyter, use: %matplotlib widget (or %matplotlib notebook for older versions)
# This enables rotatable 3D plots
try:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        ipython.run_line_magic('matplotlib', 'widget')
except:
    try:
        if ipython is not None:
            ipython.run_line_magic('matplotlib', 'notebook')
    except:
        import matplotlib
        matplotlib.use('inline')
        print("Note: Interactive 3D plots not available. Using static plots.")

In [ ]:
def load_metadata(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} does not exist")
    data = np.load(path, allow_pickle=True)
    return data

def get_slice(data, key_prefix, idx, reshape_dim=None):
    """Helper to extract a slice from ragged array storage."""
    try:
        offsets = data[f"{key_prefix}_offsets"]
        lengths = data[f"{key_prefix}_lengths"]
        start = offsets[idx]
        length = lengths[idx]
        arr = data[f"{key_prefix}_data"][start:start+length]
        if reshape_dim:
            if arr.size == 0:
                return arr.reshape(0, reshape_dim)
            return arr.reshape(-1, reshape_dim)
        return arr
    except KeyError:
        return np.array([])

def get_frame_data(data, idx, reader=None):
    """Extracts ObjectFrameData components for a specific frame index from the compressed metadata.
    
    This reconstructs the exact data structure that modular_pipeline.py creates in step():
    - pose_frontend: pose after frontend (before local opt)
    - pose_local: pose after local optimization
    - reg_curr3d, reg_key_points_idx, reg_inliers, reg_residuals: registration data
    
    Args:
        data: Loaded metadata dictionary
        idx: Frame index in the metadata array
        reader: Optional dataset reader (Ho3dReader or YcbineoatReader) for loading GT poses
    """
    # 3D points observed in camera frame (from registration)
    cur_3d = get_slice(data, 'reg_curr3d', idx, 3)
    
    # Indices of these points (Track IDs)
    cur_3d_idx = get_slice(data, 'reg_key_points_idx', idx)
    
    # Valid indices used by the registration (logged as reg_valid_idx)
    valid_idx = get_slice(data, 'reg_valid_idx', idx)
    if valid_idx.size == 0:
        # Fallback: if not present, assume all current 3D indices are valid
        valid_idx = cur_3d_idx.copy()
    
    # Inliers/Residuals from registration
    inliers = get_slice(data, 'reg_inliers', idx)
    if inliers.size > 0:
        inliers = inliers.astype(bool)
    else:
        inliers = np.ones(cur_3d.shape[0], dtype=bool) if cur_3d.shape[0] > 0 else np.array([], dtype=bool)
        
    residuals = get_slice(data, 'reg_residuals', idx)
    if residuals.size == 0:
        residuals = np.zeros(cur_3d.shape[0]) if cur_3d.shape[0] > 0 else np.array([])
    
    # Uncertainties (from track table)
    uncertainties = get_slice(data, 'uncertainties', idx)
    if uncertainties.size == 0:
        uncertainties = np.ones(cur_3d.shape[0]) * 0.1 if cur_3d.shape[0] > 0 else np.array([])
    
    # Poses at different stages
    # pose_frontend: pose after frontend registration (before local optimization)
    # This is stored in the metadata as the object pose after frontend.step()
    if 'pose_frontend' in data:
        pose_frontend = data['pose_frontend'][idx]
    else:
        # If pose_frontend doesn't exist, try to use obj_pose (final pose) as approximation
        # But this is not ideal - we should have pose_frontend in metadata
        print(f"Warning: pose_frontend not found in metadata for frame {idx}, using obj_pose")
        pose_frontend = data['obj_pose'][idx] if 'obj_pose' in data else data['obj_init_pose'][idx]
    
    # pose_local: pose after local optimization
    if 'pose_local' in data:
        pose_local = data['pose_local'][idx]
    else:
        # Fallback to pose_frontend if pose_local not available
        pose_local = pose_frontend
    
    # Frame ID
    frame_id = int(data['frame_id'][idx])
    
    # GT Pose: Load from dataset reader if available (exactly as visualize_pose_debug.py)
    gt_pose = None
    if reader is not None:
        try:
            # frame_id is the index in the dataset (not the metadata index)
            # Match visualize_pose_debug.py: check bounds and handle None returns
            if 0 <= frame_id < len(reader):
                pose = reader.get_gt_pose(frame_id)
                if pose is not None:
                    gt_pose = pose
        except Exception as e:
            # Silently fail if GT loading fails (e.g., no GT available for this frame)
            pass
    
    # Is keyframe?
    is_keyframe = data['is_key_frame'][idx] if 'is_key_frame' in data else False
    
    # Relative pose (we'll compute from previous frame)
    rel_pose = None  # Will be computed from previous frame
    
    return {
        'frame_id': frame_id,
        'pose_frontend': pose_frontend,
        'pose_local': pose_local,
        'gt_pose': gt_pose,
        'cur_3d': cur_3d,
        'cur_3d_idx': cur_3d_idx,
        'valid_idx': valid_idx,
        'inliers': inliers,
        'residuals': residuals,
        'uncertainties': uncertainties,
        'is_keyframe': is_keyframe
    }

In [ ]:
# --- CONFIGURATION ---
# Path to your metadata file
file_path = '/home/justin/code/point-to-pose/results/ho3d_single/MPM10/meta_data/meta_data.npz'

# Dataset configuration for GT pose loading
# Set these to load ground truth poses from the dataset
data_path = "/home/justin/data/HO3D_V3/evaluation"  # e.g., '/mnt/9a72c439-d0a7-45e8-8d20-d7a235d02763/DATASET/HO3D' or '/path/to/YCBInEoat'
video_name = "MPM10"  # e.g., 'MPM10' or 'bleach0'
dataset_type = "ho3d"  # 'ho3d' or 'ycbineoat' (auto-detected from data_path if None)

# Fallback for development if file doesn't exist
if not os.path.exists(file_path):
    print(f"Path {file_path} not found.")
    # Check if we have the temp debug file
    temp_path = os.path.abspath('../debug_temp/pipeline/meta_data/meata_data.npz')
    if os.path.exists(temp_path):
        print(f"Using debug file: {temp_path}")
        file_path = temp_path
    else:
        print("No metadata file found. Please adjust 'file_path'.")

print(f"Loading {file_path}...")
meta_data = load_metadata(file_path)
num_frames = len(meta_data['frame_id'])
print(f"Loaded {num_frames} frames.")

# Initialize dataset reader for GT pose loading (exactly as visualize_pose_debug.py)
gt_reader = None
if data_path is not None and video_name is not None:
    if not os.path.exists(data_path):
        print(f"Warning: Data path {data_path} does not exist. GT poses will not be loaded.")
    else:
        try:
            # Auto-detect dataset type if not specified
            if dataset_type is None:
                if 'ho3d' in data_path.lower():
                    dataset_type = 'ho3d'
                elif 'ycb' in data_path.lower() or 'ycbineoat' in data_path.lower():
                    dataset_type = 'ycbineoat'
                else:
                    print(f"Warning: Could not auto-detect dataset type from data_path. Please set dataset_type explicitly.")
                    dataset_type = None
            
            if dataset_type == 'ho3d':
                # Determine video path: root/evaluation/VIDEO_NAME or root/VIDEO_NAME
                # Handle case where data_path might be root or evaluation directory
                ho3d_root = data_path
                if os.path.basename(data_path) == 'evaluation':
                    # data_path is already the evaluation directory, go up one level for root
                    ho3d_root = os.path.dirname(data_path)
                
                video_path = os.path.join(ho3d_root, "evaluation", video_name)
                if not os.path.exists(video_path):
                    video_path = os.path.join(ho3d_root, video_name)
                
                if os.path.exists(video_path):
                    # Ho3dReader expects (video_dir, ho3d_root) - root is the HO3D dataset root
                    gt_reader = Ho3dReader(video_path, ho3d_root)
                    print(f"Loading GT for {video_name} from {data_path} (dataset: ho3d)...")
                    print(f"  Video path: {video_path}")
                    print(f"  HO3D root: {ho3d_root}")
                    print(f"  Initialized Ho3dReader with {len(gt_reader)} frames")
                else:
                    print(f"Warning: Video path {video_path} not found. GT poses will not be loaded.")
                    print(f"  Tried: {os.path.join(ho3d_root, 'evaluation', video_name)}")
                    print(f"         {os.path.join(ho3d_root, video_name)}")
            elif dataset_type == 'ycbineoat':
                # YCBInEoat layout: root/VIDEO_NAME
                video_path = os.path.join(data_path, video_name)
                if os.path.exists(video_path):
                    gt_reader = YcbineoatReader(video_path)
                    print(f"Loading GT for {video_name} from {data_path} (dataset: ycbineoat)...")
                    print(f"  Video path: {video_path}")
                    print(f"  Initialized YcbineoatReader with {len(gt_reader)} frames")
                else:
                    print(f"Warning: Video path {video_path} not found. GT poses will not be loaded.")
            else:
                print(f"Warning: Unknown dataset type '{dataset_type}'. GT poses will not be loaded.")
                print(f"  Supported types: 'ho3d', 'ycbineoat'")
        except Exception as e:
            print(f"Warning: Failed to initialize GT reader: {e}")
            import traceback
            traceback.print_exc()
            print("GT poses will not be loaded.")
else:
    print("Note: data_path and video_name not set. GT poses will not be loaded.")
    print("  To enable GT loading, set:")
    print("    data_path = '/path/to/dataset/root'  (e.g., '/path/to/HO3D' or '/path/to/YCBInEoat')")
    print("    video_name = 'VIDEO_NAME'  (e.g., 'MPM10' or 'bleach0')")
    print("    dataset_type = 'ho3d' or 'ycbineoat' (optional, auto-detected)")

# Create config structure matching pipeline (from ho3d_single.yaml)
pipeline_cfg = edict({
    'local_optimizer': edict({
        'type': 'lm_graph',
        'params': edict({
            'local_graph_max_num_frames': -1,
            'max_iterations': 20,
            'relative_error_tol': 1e-5,
            'absolute_error_tol': 1e-5,
            'prior_noise_param': [0.1, 0.1, 0.1, 0.1, 0.1, 0.1]
        })
    }),
    'global_optimizer': edict({
        'type': 'lm_graph',
        'params': edict({
            'relinearize_threshold': 0.1,
            'relinearize_skip': 1,
            'relative_error_tol': 1e-5,
            'absolute_error_tol': 1e-5,
            'prior_noise_param': [0.01, 0.01, 0.01, 0.01, 0.01, 0.01]
        })
    })
})

In [ ]:
# Initialize pipeline components (exactly as modular_pipeline.py does)
local_optimizer = LocalOptimizer(pipeline_cfg)
kf_graph = KeyFrameGraph(pipeline_cfg)

# Create a mock Object for state tracking (as pipeline does)
mock_obj = Object(0)
mock_obj.pose = np.eye(4)

# Create a mock track_table for update_object_state (simplified)
mock_track_table = PointTrackTable.new(n0=0)
# We'll populate obj2track_map as we go
mock_track_table.obj2track_map = {0: []}

# Storage for trajectories
traj_frontend = []  # Pose after frontend (before local opt)
traj_local = []     # Pose after local optimization
traj_global = []    # Pose after global optimization (keyframes only)
traj_gt = []        # Ground truth
local_errors = []   # Graph errors from local optimizer
global_errors = []  # Graph errors from global optimizer

# Track keyframes for global optimization
keyframes_list = []
kf_idx_counter = 0
keyframe_frame_ids = []  # Store frame IDs that are keyframes
keyframe_indices = []    # Store indices in trajectory array for keyframes

# --- RECREATE PIPELINE STEP LOOP ---
# This mimics modular_pipeline.py step() method
start_frame = 0
end_frame = start_frame + min(600, num_frames)  # Run first 100 frames
# end_frame = start_frame + min(90, num_frames)  # Run first 88 frames

print(f"Recreating pipeline optimization for frames {start_frame} to {end_frame}...")

for i in range(start_frame, end_frame):
    fd = get_frame_data(meta_data, i, reader=gt_reader)
    
    # Compute relative pose (as pipeline does in step())
    if i == start_frame:
        rel_pose = np.eye(4)  # First frame
    else:
        prev_fd = get_frame_data(meta_data, i-1, reader=gt_reader)
        # Pipeline uses pose_frontend for rel_pose calculation
        prev_pose = prev_fd['pose_frontend']
        cur_pose = fd['pose_frontend']
        # rel_pose = T_prev^{-1} * T_curr (as in frontend)
        prev_pose_inv = inverse_SE3(prev_pose)
        rel_pose = cur_pose @ prev_pose_inv
    
    # Store frontend pose
    traj_frontend.append(fd['pose_frontend'].copy())
    
    # --- LOCAL OPTIMIZATION (as in pipeline step()) ---
    # Create ObjectFrameData exactly as pipeline does (line 271-285)
    object_frame_data = ObjectFrameData(
        obj_id=0,
        frame_id=fd['frame_id'],
        pose=fd['pose_frontend'],  # Use frontend pose as input
        rel_pose=rel_pose,
        cur_3d=fd['cur_3d'],
        cur_3d_idx=fd['cur_3d_idx'],
        inliers=fd['inliers'],
        residuals=fd['residuals'],
        valid_idx=fd['valid_idx'],
        uncertainties=fd['uncertainties']
    )
    
    # Run local optimization (as pipeline line 287)
    # opt_result = local_optimizer.optimize(object_frame_data)
    opt_result = None
    
    # Update object state (as pipeline line 288-290)
    # Note: For debugging, we mainly care about pose optimization, so we'll update pose directly
    # The key_points update in update_object_state requires the object to have key_points initialized,
    # which is complex to maintain. Since we're debugging optimizer behavior, pose is sufficient.
    if opt_result is not None:
        # Update pose directly (this is what we're debugging)
        mock_obj.pose = opt_result.pose_optimized.copy()
        
        # Update track_table mapping (for potential future use)
        if len(fd['cur_3d_idx']) > 0:
            mock_track_table.obj2track_map[0] = fd['cur_3d_idx'].tolist()
        
        # Skip update_object_state to avoid key_points initialization issues
        # The optimizer result already contains the optimized pose, which is what we need
        
        traj_local.append(mock_obj.pose.copy())
        
        # Get graph error from underlying optimizer
        underlying_opt = local_optimizer._get_optimizer(0)
        if hasattr(underlying_opt, '_graph') and hasattr(underlying_opt, '_values'):
            err = underlying_opt._graph.error(underlying_opt._values)
            local_errors.append(err)
        else:
            local_errors.append(0.0)
    else:
        # If optimization failed, use frontend pose and update mock_obj for consistency
        # print(f"Local optimization failed for frame {fd['frame_id']}")
        
        mock_obj.pose = fd['pose_frontend'].copy()
        traj_local.append(fd['pose_frontend'].copy())
        local_errors.append(0.0)
    
    # --- KEYFRAME HANDLING (as in pipeline) ---
    if fd['is_keyframe']:
        # Track this as a keyframe
        keyframe_frame_ids.append(fd['frame_id'])
        keyframe_indices.append(i - start_frame)  # Index in trajectory array
        
        # Create KeyFrame object (simplified - we don't have all KF data in metadata)
        # But we can reconstruct from what we have
        # Important: All arrays must have consistent sizes for the observed points
        
        # Use the registration data (cur_3d, cur_3d_idx) as observed points
        num_obs = len(fd['cur_3d'])
        
        # Ensure uncertainties array matches the size of observed points
        if len(fd['uncertainties'][fd['cur_3d_idx']]) == num_obs:
            obs_uncertainties = fd['uncertainties'][fd['cur_3d_idx']]
        elif len(fd['uncertainties']) > 0:
            # If sizes don't match, use a default value or take first N
            obs_uncertainties = np.ones(num_obs, dtype=float) * 0.1
            print(f"Warning: Frame {fd['frame_id']} - uncertainties size mismatch, using defaults")
        else:
            obs_uncertainties = np.ones(num_obs, dtype=float) * 0.1
        
        kf = KeyFrame(
            frame_id=fd['frame_id'],
            obj_id=0,
            kf_idx=kf_idx_counter,
            timestamp=None,
            pose=mock_obj.pose.copy(),  # Use optimized pose
            kp_track_indices=fd['cur_3d_idx'] if num_obs > 0 else np.array([], dtype=int),
            kp_2d=np.zeros((num_obs, 2)) if num_obs > 0 else np.zeros((0, 2)),
            kp_3d_camera=fd['cur_3d'] if num_obs > 0 else np.zeros((0, 3)),
            kp_3d_object=np.zeros((num_obs, 3)) if num_obs > 0 else np.zeros((0, 3)),
            kp_valid=np.ones(num_obs, dtype=bool) if num_obs > 0 else np.array([], dtype=bool),
            obs_track_indices=fd['cur_3d_idx'] if num_obs > 0 else np.array([], dtype=int),
            obs_2d=np.zeros((num_obs, 2)) if num_obs > 0 else np.zeros((0, 2)),
            obs_3d_camera=fd['cur_3d'] if num_obs > 0 else np.zeros((0, 3)),
            obs_3d_object=np.zeros((num_obs, 3)) if num_obs > 0 else np.zeros((0, 3)),
            obs_valid=np.ones(num_obs, dtype=bool) if num_obs > 0 else np.array([], dtype=bool),
            obs_visible=np.ones(num_obs, dtype=bool) if num_obs > 0 else np.array([], dtype=bool),
            obs_uncertainties=obs_uncertainties,
            reg_curr3d=fd['cur_3d'],
            reg_inliers=fd['inliers'],
            reg_residuals=fd['residuals'],
            reg_valid_idx=fd['valid_idx'],
            dense_pts=np.zeros((0, 3))
        )
        keyframes_list.append(kf)
        kf_idx_counter += 1
        
        # Reset local optimizer on keyframe (as pipeline line 312)
        local_optimizer.reset(0)
    
    # --- GLOBAL OPTIMIZATION (as in pipeline line 319-329) ---
    # Process keyframes in batch (pipeline processes new_keyframes)
    global_pose_updated = False

    print(f"frame {i}, fd['is_keyframe']: {fd['is_keyframe']}; keyframes_list: {len(keyframes_list)}")
    if keyframes_list:
        updated_poses, updated_landmarks = kf_graph.update(keyframes_list)
        
        # Update object pose from global optimization
        # Note: Global optimization only updates keyframe poses, not every frame
        for kf in keyframes_list:
            key = (kf.obj_id, kf.kf_idx)
            if key in updated_poses:
                mock_obj.pose = updated_poses[key].copy()
                global_pose_updated = True
        
        # Get graph error from global optimizer
        underlying_global_opt = kf_graph._get_optimizer(0)
        if hasattr(underlying_global_opt, '_graph') and hasattr(underlying_global_opt, '_values'):
            err = underlying_global_opt._graph.error(underlying_global_opt._values)
            global_errors.append(err)
        else:
            global_errors.append(0.0)
        
        keyframes_list = []  # Clear after processing
    
    # Store global pose
    # For keyframes, this will be the globally optimized pose
    # For non-keyframes, this will be the same as local (since global only optimizes keyframes)
    traj_global.append(mock_obj.pose.copy())
    
    # Store GT
    if fd['gt_pose'] is not None:
        traj_gt.append(fd['gt_pose'].copy())
    else:
        if len(traj_gt) == 0:
            traj_gt.append(np.eye(4))
        else:
            traj_gt.append(traj_gt[-1].copy())

print(f"Completed optimization loop. Processed {end_frame - start_frame} frames.")
print(f"Found {kf_idx_counter} keyframes.")
print(f"\n=== Keyframe Information ===")
print(f"Keyframe Frame IDs: {keyframe_frame_ids}")
print(f"Keyframe Trajectory Indices: {keyframe_indices}")
if len(keyframe_frame_ids) > 0:
    print(f"First keyframe at frame {keyframe_frame_ids[0]} (trajectory index {keyframe_indices[0]})")
    print(f"Last keyframe at frame {keyframe_frame_ids[-1]} (trajectory index {keyframe_indices[-1]})")

# Debug GT pose loading
print(f"\n=== GT Pose Loading Debug ===")
gt_valid_count = sum(1 for i in range(start_frame, end_frame) 
                     if get_frame_data(meta_data, i, reader=gt_reader)['gt_pose'] is not None)
print(f"GT poses loaded: {gt_valid_count} / {end_frame - start_frame} frames")

# Check first few frames
print(f"\nFirst 5 frames GT pose check:")
for i in range(min(5, end_frame - start_frame)):
    fd = get_frame_data(meta_data, i, reader=gt_reader)
    frame_id = fd['frame_id']
    gt_pose = fd['gt_pose']
    pred_pose = fd['pose_local']
    if gt_pose is not None:
        gt_t = gt_pose[:3, 3]
        pred_t = pred_pose[:3, 3]
        diff_t = np.linalg.norm(gt_t - pred_t)
        print(f"  Frame {i} (dataset frame_id={frame_id}): GT_t={gt_t}, Pred_t={pred_t}, Diff={diff_t:.4f}m")
    else:
        print(f"  Frame {i} (dataset frame_id={frame_id}): No GT available")

# Optional: Align GT to first valid prediction (similar to visualize_pose_debug.py --align_gt)
align_gt = True  # Set to False to disable alignment
if align_gt and gt_reader is not None:
    # Find first frame with valid GT
    first_valid_idx = None
    first_valid_gt = None
    for i in range(start_frame, end_frame):
        fd = get_frame_data(meta_data, i, reader=gt_reader)
        if fd['gt_pose'] is not None:
            first_valid_idx = i
            first_valid_gt = fd['gt_pose'].copy()
            break
    
    if first_valid_idx is not None and first_valid_gt is not None:
        # Compute alignment transform: T_align = pred[first] @ inv(gt[first])
        # Use global pose (or local if global not available) for alignment
        pred_pose_first = traj_global[first_valid_idx - start_frame]
        T_align = pred_pose_first @ inverse_SE3(first_valid_gt)
        
        # Reload and apply alignment to all GT poses
        traj_gt_aligned = []
        for i in range(start_frame, end_frame):
            fd = get_frame_data(meta_data, i, reader=gt_reader)
            if fd['gt_pose'] is not None:
                traj_gt_aligned.append(fd['gt_pose'] @ T_align)
            else:
                # Keep previous value (identity or last valid)
                if len(traj_gt_aligned) == 0:
                    traj_gt_aligned.append(np.eye(4))
                else:
                    traj_gt_aligned.append(traj_gt_aligned[-1].copy())
        traj_gt = traj_gt_aligned
        
        print(f"\nAligned GT to first valid frame (metadata idx={first_valid_idx})")
        print(f"  Alignment transform translation: {T_align[:3, 3]}")
        print(f"  Before alignment - GT_t: {first_valid_gt[:3, 3]}, Pred_t: {pred_pose_first[:3, 3]}")
        print(f"  After alignment - GT_t: {(first_valid_gt @ T_align)[:3, 3]}, Pred_t: {pred_pose_first[:3, 3]}")
    else:
        print("\nNo valid GT frames found for alignment")

# Store keyframe info for later use
KEYFRAME_FRAME_IDS = keyframe_frame_ids
KEYFRAME_INDICES = keyframe_indices

In [ ]:
# fd = get_frame_data(meta_data,230)
# print(fd.keys())
# print(fd['cur_3d'].shape)
# print(fd['cur_3d_idx'].shape)
# print(fd['uncertainties'].shape)
# print(fd['residuals'].shape)
# print(fd['inliers'].shape)
# print(fd['pose_frontend'].shape)
# print(fd['uncertainties'])

# print(fd['cur_3d_idx'])

# print(fd['uncertainties'][fd['cur_3d_idx']].shape)




In [ ]:
# Helper function to extract Euler angles (roll, pitch, yaw) from rotation matrix
def rotation_matrix_to_euler(rot_mat):
    """Convert rotation matrix to roll, pitch, yaw (in degrees)"""
    r = R.from_matrix(rot_mat)
    euler = r.as_euler('xyz', degrees=True)
    return euler

# --- VISUALIZATION ---
traj_frontend = np.array(traj_frontend)
traj_local = np.array(traj_local)
traj_global = np.array(traj_global)
traj_gt = np.array(traj_gt)

# Debug: Check if trajectories are populated
print(f"\n=== Trajectory Data Check ===")
print(f"Frontend trajectory shape: {traj_frontend.shape}")
print(f"Local trajectory shape: {traj_local.shape}")
print(f"Global trajectory shape: {traj_global.shape}")
print(f"GT trajectory shape: {traj_gt.shape}")

# Check if trajectories differ
if len(traj_local) > 0 and len(traj_global) > 0:
    local_global_diff = np.linalg.norm(traj_local[:, :3, 3] - traj_global[:, :3, 3], axis=1)
    num_different = np.sum(local_global_diff > 1e-6)
    print(f"Frames where local != global: {num_different} / {len(traj_local)}")
    if num_different > 0:
        print(f"Max difference: {np.max(local_global_diff):.6f} m")
    else:
        print("Note: Local and Global trajectories are identical (global only optimizes keyframes)")

# Check frontend vs local differences
if len(traj_frontend) > 0 and len(traj_local) > 0:
    frontend_local_diff = np.linalg.norm(traj_frontend[:, :3, 3] - traj_local[:, :3, 3], axis=1)
    num_different_fe = np.sum(frontend_local_diff > 1e-6)
    print(f"\nFrames where frontend != local: {num_different_fe} / {len(traj_frontend)}")
    if num_different_fe > 0:
        print(f"Max frontend-local difference: {np.max(frontend_local_diff):.6f} m")
        print(f"Mean frontend-local difference: {np.mean(frontend_local_diff):.6f} m")
    else:
        print("Warning: Frontend and Local trajectories are identical - frontend may not be stored correctly in metadata")
    
    # Check first few frames to see if frontend is actually different
    print(f"\nFirst 5 frames translation comparison:")
    for i in range(min(5, len(traj_frontend))):
        fe_t = traj_frontend[i, :3, 3]
        loc_t = traj_local[i, :3, 3]
        diff = np.linalg.norm(fe_t - loc_t)
        print(f"  Frame {i}: Frontend={fe_t}, Local={loc_t}, Diff={diff:.6f} m")

# 1. 3D Trajectory Comparison (Interactive/Rotatable)
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
# Make frontend more visible with distinct style
ax.plot(traj_frontend[:, 0, 3], traj_frontend[:, 1, 3], traj_frontend[:, 2, 3], 
        'r.-', alpha=0.8, label='Frontend', linewidth=2, markersize=4, markevery=5)
ax.plot(traj_local[:, 0, 3], traj_local[:, 1, 3], traj_local[:, 2, 3], 
        'b.-', alpha=0.7, label='Local Opt', linewidth=1.5, markersize=3, markevery=10)
# Global optimization: only show at keyframes as dots
if len(keyframe_indices) > 0:
    kf_idx_array = np.array(keyframe_indices)
    ax.scatter(traj_global[kf_idx_array, 0, 3], 
               traj_global[kf_idx_array, 1, 3], 
               traj_global[kf_idx_array, 2, 3], 
               c='m', marker='o', s=100, alpha=0.9, label='Global Opt (Keyframes)', 
               edgecolors='darkmagenta', linewidths=1.5, zorder=5)
if len(traj_gt) > 0:
    ax.plot(traj_gt[:, 0, 3], traj_gt[:, 1, 3], traj_gt[:, 2, 3], 
            'g.-', label='GT', linewidth=2.5, markersize=4, markevery=5)

ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_zlabel('Z (m)')
ax.legend()
ax.set_title("3D Trajectory: Pipeline Recreation (Rotatable)")
plt.show()

# 2. Translation Components (X, Y, Z)
fig, axes = plt.subplots(3, 1, figsize=(15, 10))
frames = np.arange(len(traj_frontend))

# Translation X
axes[0].plot(frames, traj_frontend[:, 0, 3], 'r-', alpha=0.8, label='Frontend', linewidth=2.5, marker='D', markersize=3, markevery=10)
axes[0].plot(frames, traj_local[:, 0, 3], 'b-', label='Local Opt', linewidth=2, marker='o', markersize=2, markevery=10)
# Global optimization: only show at keyframes as dots
if len(keyframe_indices) > 0:
    kf_idx_array = np.array(keyframe_indices)
    axes[0].scatter(frames[kf_idx_array], traj_global[kf_idx_array, 0, 3], 
                    c='m', marker='o', s=100, alpha=0.9, label='Global Opt (Keyframes)',
                    edgecolors='darkmagenta', linewidths=1.5, zorder=5)
if len(traj_gt) > 0:
    axes[0].plot(frames, traj_gt[:, 0, 3], 'g-', label='GT', linewidth=2.5)
axes[0].set_title("Translation X over Time")
axes[0].set_xlabel("Frame")
axes[0].set_ylabel("X (m)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Translation Y
axes[1].plot(frames, traj_frontend[:, 1, 3], 'r-', alpha=0.8, label='Frontend', linewidth=2.5, marker='D', markersize=3, markevery=10)
axes[1].plot(frames, traj_local[:, 1, 3], 'b-', label='Local Opt', linewidth=2, marker='o', markersize=2, markevery=10)
# Global optimization: only show at keyframes as dots
if len(keyframe_indices) > 0:
    kf_idx_array = np.array(keyframe_indices)
    axes[1].scatter(frames[kf_idx_array], traj_global[kf_idx_array, 1, 3], 
                    c='m', marker='o', s=100, alpha=0.9, label='Global Opt (Keyframes)',
                    edgecolors='darkmagenta', linewidths=1.5, zorder=5)
if len(traj_gt) > 0:
    axes[1].plot(frames, traj_gt[:, 1, 3], 'g-', label='GT', linewidth=2.5)
axes[1].set_title("Translation Y over Time")
axes[1].set_xlabel("Frame")
axes[1].set_ylabel("Y (m)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Translation Z
axes[2].plot(frames, traj_frontend[:, 2, 3], 'r-', alpha=0.8, label='Frontend', linewidth=2.5, marker='D', markersize=3, markevery=10)
axes[2].plot(frames, traj_local[:, 2, 3], 'b-', label='Local Opt', linewidth=2, marker='o', markersize=2, markevery=10)
# Global optimization: only show at keyframes as dots
if len(keyframe_indices) > 0:
    kf_idx_array = np.array(keyframe_indices)
    axes[2].scatter(frames[kf_idx_array], traj_global[kf_idx_array, 2, 3], 
                    c='m', marker='o', s=100, alpha=0.9, label='Global Opt (Keyframes)',
                    edgecolors='darkmagenta', linewidths=1.5, zorder=5)
if len(traj_gt) > 0:
    axes[2].plot(frames, traj_gt[:, 2, 3], 'g-', label='GT', linewidth=2.5)
axes[2].set_title("Translation Z over Time")
axes[2].set_xlabel("Frame")
axes[2].set_ylabel("Z (m)")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 3. Rotation Components (Roll, Pitch, Yaw)
fig, axes = plt.subplots(3, 1, figsize=(15, 10))

# Extract Euler angles
roll_frontend, pitch_frontend, yaw_frontend = [], [], []
roll_local, pitch_local, yaw_local = [], [], []
roll_global, pitch_global, yaw_global = [], [], []
roll_gt, pitch_gt, yaw_gt = [], [], []

for i in range(len(traj_frontend)):
    euler_f = rotation_matrix_to_euler(traj_frontend[i, :3, :3])
    euler_l = rotation_matrix_to_euler(traj_local[i, :3, :3])
    euler_g = rotation_matrix_to_euler(traj_global[i, :3, :3])
    
    roll_frontend.append(euler_f[0])
    pitch_frontend.append(euler_f[1])
    yaw_frontend.append(euler_f[2])
    
    roll_local.append(euler_l[0])
    pitch_local.append(euler_l[1])
    yaw_local.append(euler_l[2])
    
    roll_global.append(euler_g[0])
    pitch_global.append(euler_g[1])
    yaw_global.append(euler_g[2])
    
    if len(traj_gt) > 0:
        euler_gt = rotation_matrix_to_euler(traj_gt[i, :3, :3])
        roll_gt.append(euler_gt[0])
        pitch_gt.append(euler_gt[1])
        yaw_gt.append(euler_gt[2])

# Roll
axes[0].plot(frames, roll_frontend, 'r-', alpha=0.8, label='Frontend', linewidth=2.5, marker='D', markersize=3, markevery=10)
axes[0].plot(frames, roll_local, 'b-', label='Local Opt', linewidth=2, marker='o', markersize=2, markevery=10)
# Global optimization: only show at keyframes as dots
if len(keyframe_indices) > 0:
    kf_idx_array = np.array(keyframe_indices)
    axes[0].scatter(frames[kf_idx_array], [roll_global[i] for i in kf_idx_array], 
                    c='m', marker='o', s=100, alpha=0.9, label='Global Opt (Keyframes)',
                    edgecolors='darkmagenta', linewidths=1.5, zorder=5)
if len(traj_gt) > 0:
    axes[0].plot(frames, roll_gt, 'g-', label='GT', linewidth=2.5)
axes[0].set_title("Rotation Roll (X-axis) over Time")
axes[0].set_xlabel("Frame")
axes[0].set_ylabel("Roll (degrees)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Pitch
axes[1].plot(frames, pitch_frontend, 'r-', alpha=0.8, label='Frontend', linewidth=2.5, marker='D', markersize=3, markevery=10)
axes[1].plot(frames, pitch_local, 'b-', label='Local Opt', linewidth=2, marker='o', markersize=2, markevery=10)
# Global optimization: only show at keyframes as dots
if len(keyframe_indices) > 0:
    kf_idx_array = np.array(keyframe_indices)
    axes[1].scatter(frames[kf_idx_array], [pitch_global[i] for i in kf_idx_array], 
                    c='m', marker='o', s=100, alpha=0.9, label='Global Opt (Keyframes)',
                    edgecolors='darkmagenta', linewidths=1.5, zorder=5)
if len(traj_gt) > 0:
    axes[1].plot(frames, pitch_gt, 'g-', label='GT', linewidth=2.5)
axes[1].set_title("Rotation Pitch (Y-axis) over Time")
axes[1].set_xlabel("Frame")
axes[1].set_ylabel("Pitch (degrees)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Yaw
axes[2].plot(frames, yaw_frontend, 'r-', alpha=0.8, label='Frontend', linewidth=2.5, marker='D', markersize=3, markevery=10)
axes[2].plot(frames, yaw_local, 'b-', label='Local Opt', linewidth=2, marker='o', markersize=2, markevery=10)
# Global optimization: only show at keyframes as dots
if len(keyframe_indices) > 0:
    kf_idx_array = np.array(keyframe_indices)
    axes[2].scatter(frames[kf_idx_array], [yaw_global[i] for i in kf_idx_array], 
                    c='m', marker='o', s=100, alpha=0.9, label='Global Opt (Keyframes)',
                    edgecolors='darkmagenta', linewidths=1.5, zorder=5)
if len(traj_gt) > 0:
    axes[2].plot(frames, yaw_gt, 'g-', label='GT', linewidth=2.5)
axes[2].set_title("Rotation Yaw (Z-axis) over Time")
axes[2].set_xlabel("Frame")
axes[2].set_ylabel("Yaw (degrees)")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 4. Additional Analysis Plots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# X-Y Plane
axes[0,0].plot(traj_frontend[:, 0, 3], traj_frontend[:, 1, 3], 'r.-', alpha=0.8, label='Frontend', linewidth=2, markersize=4, markevery=5)
axes[0,0].plot(traj_local[:, 0, 3], traj_local[:, 1, 3], 'b.-', alpha=0.7, label='Local Opt', linewidth=1.5, markersize=3, markevery=10)
# Global optimization: only show at keyframes as dots
if len(keyframe_indices) > 0:
    kf_idx_array = np.array(keyframe_indices)
    axes[0,0].scatter(traj_global[kf_idx_array, 0, 3], traj_global[kf_idx_array, 1, 3], 
                      c='m', marker='o', s=100, alpha=0.9, label='Global Opt (Keyframes)',
                      edgecolors='darkmagenta', linewidths=1.5, zorder=5)
if len(traj_gt) > 0:
    axes[0,0].plot(traj_gt[:, 0, 3], traj_gt[:, 1, 3], 'g.-', label='GT', linewidth=2.5, markersize=4, markevery=5)
axes[0,0].set_title("X-Y Plane")
axes[0,0].set_xlabel("X (m)")
axes[0,0].set_ylabel("Y (m)")
axes[0,0].axis('equal')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Local Graph Error
if len(local_errors) > 0:
    axes[0,1].plot(local_errors, 'b-')
    axes[0,1].set_title("Local Graph Error")
    axes[0,1].set_xlabel("Frame")
    axes[0,1].set_ylabel("Error")
    axes[0,1].set_yscale('log')
    axes[0,1].grid(True, alpha=0.3)

# Global Graph Error
if len(global_errors) > 0:
    axes[1,0].plot(global_errors, 'm-')
    axes[1,0].set_title("Global Graph Error (Keyframes)")
    axes[1,0].set_xlabel("Keyframe")
    axes[1,0].set_ylabel("Error")
    axes[1,0].set_yscale('log')
    axes[1,0].grid(True, alpha=0.3)

# Rotation Error vs GT
rot_diffs_local = []
rot_diffs_global = []
if len(traj_gt) > 0:
    for i in range(len(traj_frontend)):
        R_gt = traj_gt[i, :3, :3]
        R_local = traj_local[i, :3, :3]
        R_global = traj_global[i, :3, :3]
        
        # Local error
        R_diff_local = R_gt.T @ R_local
        tr_local = np.trace(R_diff_local)
        theta_local = np.arccos(np.clip((tr_local - 1)/2, -1, 1))
        rot_diffs_local.append(np.degrees(theta_local))
        
        # Global error
        R_diff_global = R_gt.T @ R_global
        tr_global = np.trace(R_diff_global)
        theta_global = np.arccos(np.clip((tr_global - 1)/2, -1, 1))
        rot_diffs_global.append(np.degrees(theta_global))
    
    axes[1,1].plot(rot_diffs_local, 'b-', label='Local vs GT', alpha=0.7)
    axes[1,1].plot(rot_diffs_global, 'm-', label='Global vs GT', alpha=0.7)
    axes[1,1].set_title("Rotation Error vs GT (degrees)")
    axes[1,1].set_xlabel("Frame")
    axes[1,1].set_ylabel("Angle Error (deg)")
    axes[1,1].legend()
    axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistics
print("\n=== Optimization Statistics ===")
if len(traj_gt) > 0:
    # Translation errors
    trans_err_local = np.linalg.norm(traj_local[:, :3, 3] - traj_gt[:, :3, 3], axis=1)
    trans_err_global = np.linalg.norm(traj_global[:, :3, 3] - traj_gt[:, :3, 3], axis=1)
    trans_err_frontend = np.linalg.norm(traj_frontend[:, :3, 3] - traj_gt[:, :3, 3], axis=1)
    print(f"Mean Translation Error - Frontend: {np.mean(trans_err_frontend):.4f} m")
    print(f"Mean Translation Error - Local: {np.mean(trans_err_local):.4f} m")
    print(f"Mean Translation Error - Global: {np.mean(trans_err_global):.4f} m")
    print(f"Mean Rotation Error - Local: {np.mean(rot_diffs_local):.2f} deg")
    print(f"Mean Rotation Error - Global: {np.mean(rot_diffs_global):.2f} deg")

# 5. Detailed Error Analysis: Translation and Rotation Errors (per component)
# 6 separate plots: X, Y, Z, Roll, Pitch, Yaw (one row per plot)
fig, axes = plt.subplots(6, 1, figsize=(16, 18))

if len(traj_gt) > 0:
    # Compute translation errors
    trans_errors_frontend = traj_frontend[:, :3, 3] - traj_gt[:, :3, 3]
    trans_errors_local = traj_local[:, :3, 3] - traj_gt[:, :3, 3]
    trans_errors_global = traj_global[:, :3, 3] - traj_gt[:, :3, 3]
    
    # Compute rotation errors
    def pose_to_euler(pose):
        R_mat = pose[:3, :3]
        # Ensure valid rotation matrix
        U, s, Vt = np.linalg.svd(R_mat)
        R_mat_fixed = U @ Vt
        if np.linalg.det(R_mat_fixed) < 0:
            U[:, -1] *= -1
            R_mat_fixed = U @ Vt
        try:
            return R.from_matrix(R_mat_fixed).as_euler('xyz')
        except ValueError:
            # Fallback: return zeros if still invalid
            return np.array([0.0, 0.0, 0.0])
    
    euler_gt = np.array([pose_to_euler(p) for p in traj_gt])
    euler_frontend = np.array([pose_to_euler(p) for p in traj_frontend])
    euler_local = np.array([pose_to_euler(p) for p in traj_local])
    euler_global = np.array([pose_to_euler(p) for p in traj_global])
    
    rot_errors_frontend = euler_frontend - euler_gt
    rot_errors_local = euler_local - euler_gt
    rot_errors_global = euler_global - euler_gt
    
    # Wrap angles to [-pi, pi]
    rot_errors_frontend = np.arctan2(np.sin(rot_errors_frontend), np.cos(rot_errors_frontend))
    rot_errors_local = np.arctan2(np.sin(rot_errors_local), np.cos(rot_errors_local))
    rot_errors_global = np.arctan2(np.sin(rot_errors_global), np.cos(rot_errors_global))
    
    kf_idx_array = np.array(keyframe_indices) if len(keyframe_indices) > 0 else np.array([])
    
    # Translation X
    ax = axes[0]
    ax.plot(trans_errors_frontend[:, 0], 'r-', label='Frontend', alpha=0.7, linewidth=2, marker='o', markersize=3, markevery=10)
    ax.plot(trans_errors_local[:, 0], 'b--', label='Local', alpha=0.7, linewidth=2, marker='o', markersize=3, markevery=10)
    if len(kf_idx_array) > 0:
        ax.scatter(kf_idx_array, trans_errors_global[kf_idx_array, 0], 
                  c='m', marker='o', s=60, alpha=0.8, label='Global (Keyframes)', 
                  edgecolors='darkmagenta', linewidths=1.5, zorder=5)
    ax.set_xlabel('Frame ID')
    ax.set_ylabel('Translation Error X (m)')
    ax.set_title('Translation Error X')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Translation Y
    ax = axes[1]
    ax.plot(trans_errors_frontend[:, 1], 'r-', label='Frontend', alpha=0.7, linewidth=2, marker='s', markersize=3, markevery=10)
    ax.plot(trans_errors_local[:, 1], 'b--', label='Local', alpha=0.7, linewidth=2, marker='s', markersize=3, markevery=10)
    if len(kf_idx_array) > 0:
        ax.scatter(kf_idx_array, trans_errors_global[kf_idx_array, 1], 
                  c='m', marker='o', s=60, alpha=0.8, label='Global (Keyframes)', 
                  edgecolors='darkmagenta', linewidths=1.5, zorder=5)
    ax.set_xlabel('Frame ID')
    ax.set_ylabel('Translation Error Y (m)')
    ax.set_title('Translation Error Y')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Translation Z
    ax = axes[2]
    ax.plot(trans_errors_frontend[:, 2], 'r-', label='Frontend', alpha=0.7, linewidth=2, marker='^', markersize=3, markevery=10)
    ax.plot(trans_errors_local[:, 2], 'b--', label='Local', alpha=0.7, linewidth=2, marker='^', markersize=3, markevery=10)
    if len(kf_idx_array) > 0:
        ax.scatter(kf_idx_array, trans_errors_global[kf_idx_array, 2], 
                  c='m', marker='o', s=60, alpha=0.8, label='Global (Keyframes)', 
                  edgecolors='darkmagenta', linewidths=1.5, zorder=5)
    ax.set_xlabel('Frame ID')
    ax.set_ylabel('Translation Error Z (m)')
    ax.set_title('Translation Error Z')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Rotation Roll
    ax = axes[3]
    ax.plot(rot_errors_frontend[:, 0], 'r-', label='Frontend', alpha=0.7, linewidth=2, marker='o', markersize=3, markevery=10)
    ax.plot(rot_errors_local[:, 0], 'b--', label='Local', alpha=0.7, linewidth=2, marker='o', markersize=3, markevery=10)
    if len(kf_idx_array) > 0:
        ax.scatter(kf_idx_array, rot_errors_global[kf_idx_array, 0], 
                  c='m', marker='o', s=60, alpha=0.8, label='Global (Keyframes)', 
                  edgecolors='darkmagenta', linewidths=1.5, zorder=5)
    ax.set_xlabel('Frame ID')
    ax.set_ylabel('Rotation Error Roll (rad)')
    ax.set_title('Rotation Error Roll')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Rotation Pitch
    ax = axes[4]
    ax.plot(rot_errors_frontend[:, 1], 'r-', label='Frontend', alpha=0.7, linewidth=2, marker='s', markersize=3, markevery=10)
    ax.plot(rot_errors_local[:, 1], 'b--', label='Local', alpha=0.7, linewidth=2, marker='s', markersize=3, markevery=10)
    if len(kf_idx_array) > 0:
        ax.scatter(kf_idx_array, rot_errors_global[kf_idx_array, 1], 
                  c='m', marker='o', s=60, alpha=0.8, label='Global (Keyframes)', 
                  edgecolors='darkmagenta', linewidths=1.5, zorder=5)
    ax.set_xlabel('Frame ID')
    ax.set_ylabel('Rotation Error Pitch (rad)')
    ax.set_title('Rotation Error Pitch')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Rotation Yaw
    ax = axes[5]
    ax.plot(rot_errors_frontend[:, 2], 'r-', label='Frontend', alpha=0.7, linewidth=2, marker='^', markersize=3, markevery=10)
    ax.plot(rot_errors_local[:, 2], 'b--', label='Local', alpha=0.7, linewidth=2, marker='^', markersize=3, markevery=10)
    if len(kf_idx_array) > 0:
        ax.scatter(kf_idx_array, rot_errors_global[kf_idx_array, 2], 
                  c='m', marker='o', s=60, alpha=0.8, label='Global (Keyframes)', 
                  edgecolors='darkmagenta', linewidths=1.5, zorder=5)
    ax.set_xlabel('Frame ID')
    ax.set_ylabel('Rotation Error Yaw (rad)')
    ax.set_title('Rotation Error Yaw')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Available Keyframe IDs

Below are all the keyframe frame IDs that were processed during optimization. Use these IDs in the visualization below.


In [ ]:
# Display all available keyframe IDs
print("=" * 60)
print("AVAILABLE KEYFRAME IDs")
print("=" * 60)

if len(KEYFRAME_FRAME_IDS) > 0:
    print(f"\nTotal number of keyframes: {len(KEYFRAME_FRAME_IDS)}")
    print(f"\nKeyframe Frame IDs:")
    
    # Display in columns for better readability
    cols = 5
    for i in range(0, len(KEYFRAME_FRAME_IDS), cols):
        row_ids = KEYFRAME_FRAME_IDS[i:i+cols]
        row_str = "  ".join([f"{kf_id:4d}" for kf_id in row_ids])
        print(f"  {row_str}")
    
    print(f"\nKeyframe Trajectory Indices (for reference):")
    for i in range(0, len(KEYFRAME_INDICES), cols):
        row_indices = KEYFRAME_INDICES[i:i+cols]
        row_str = "  ".join([f"{idx:4d}" for idx in row_indices])
        print(f"  {row_str}")
    
    print(f"\nTo visualize a keyframe, use one of the frame IDs above.")
    print(f"Example: visualize_keyframe({KEYFRAME_FRAME_IDS[0]})")
else:
    print("No keyframes found in the processed data.")


## Keyframe Visualization

Visualize landmarks and camera poses for a specific keyframe before and after global optimization.


In [ ]:
def visualize_keyframe(kf_frame_id):
    """
    Visualize ALL landmarks in the graph and camera poses for a specific keyframe.
    Shows landmark positions before and after optimization with arrows.
    
    Args:
        kf_frame_id: Frame ID of the keyframe to visualize
    """
    # Find the keyframe in our processed data
    if kf_frame_id not in KEYFRAME_FRAME_IDS:
        print(f"Error: Frame ID {kf_frame_id} is not a keyframe.")
        print(f"Available keyframe IDs: {KEYFRAME_FRAME_IDS}")
        return
    
    # Get the trajectory index for this keyframe
    kf_idx_in_list = KEYFRAME_FRAME_IDS.index(kf_frame_id)
    traj_idx = KEYFRAME_INDICES[kf_idx_in_list]
    
    print(f"Visualizing Keyframe: Frame ID {kf_frame_id} (Trajectory Index {traj_idx})")
    
    # Get the frame data
    frame_idx_in_meta = None
    for i in range(start_frame, end_frame):
        fd = get_frame_data(meta_data, i)
        if fd['frame_id'] == kf_frame_id:
            frame_idx_in_meta = i
            break
    
    if frame_idx_in_meta is None:
        print(f"Error: Could not find frame {kf_frame_id} in metadata.")
        return
    
    fd = get_frame_data(meta_data, frame_idx_in_meta)
    
    # Get global optimizer
    global_opt = kf_graph._get_optimizer(0)
    
    if not hasattr(global_opt, '_graph') or not hasattr(global_opt, '_values'):
        print("Error: Global optimizer graph not available.")
        return
    
    # Find the keyframe index in the global graph
    kf_idx_in_graph = kf_idx_in_list
    frame_sym = gtsam.symbol('x', kf_idx_in_graph)
    
    # Check if this keyframe exists in the global graph
    if not global_opt._values.exists(frame_sym):
        print(f"Warning: Keyframe {kf_frame_id} (kf_idx={kf_idx_in_graph}) not found in global graph.")
        print(f"Trying to find it by searching all poses...")
        found = False
        for i in range(global_opt.get_num_poses()):
            test_sym = gtsam.symbol('x', i)
            if global_opt._values.exists(test_sym):
                test_pose = global_opt._values.atPose3(test_sym)
                test_pose_matrix = inverse_SE3(test_pose.matrix())
                stored_pose = traj_global[traj_idx]
                diff = np.linalg.norm(test_pose_matrix[:3, 3] - stored_pose[:3, 3])
                if diff < 0.01:
                    frame_sym = test_sym
                    kf_idx_in_graph = i
                    found = True
                    print(f"Found keyframe at graph index {i}")
                    break
        if not found:
            print("Could not locate keyframe in global graph.")
            return
    
    # Get optimized pose from global graph
    opt_pose_gtsam = global_opt._values.atPose3(frame_sym)
    opt_pose = inverse_SE3(opt_pose_gtsam.matrix())
    init_pose = traj_local[traj_idx] if traj_idx < len(traj_local) else fd['pose_frontend']
    
    # ===== GET ALL LANDMARKS IN THE GRAPH =====
    print("Collecting ALL landmarks from global graph...")
    all_landmarks = []
    landmark_ids_seen = set()
    
    # Get all landmark IDs from the graph
    graph = global_opt._graph
    for i in range(graph.size()):
        factor = graph.at(i)
        keys = factor.keys()
        for key in keys:
            if gtsam.symbolChr(key) == ord('l'):
                lid = gtsam.symbolIndex(key)
                if lid not in landmark_ids_seen:
                    landmark_ids_seen.add(lid)
                    lm_sym = gtsam.symbol('l', int(lid))
                    if global_opt._values.exists(lm_sym):
                        lm_opt = global_opt._values.atPoint3(lm_sym)
                        all_landmarks.append({
                            'id': lid,
                            'optimized': np.array(lm_opt),
                            'initial': None  # Will fill this in
                        })
    
    print(f"Found {len(all_landmarks)} total landmarks in the graph")
    
    # ===== GET INITIAL POSITIONS FOR ALL LANDMARKS =====
    # For each landmark, find when it was first observed and get its initial position
    print("Computing initial landmark positions...")
    for lm in all_landmarks:
        lid = lm['id']
        # Search through all frames to find when this landmark was first observed
        for i in range(start_frame, min(end_frame, len(meta_data['frame_id']))):
            frame_fd = get_frame_data(meta_data, i)
            if len(frame_fd['cur_3d_idx']) > 0:
                obs_idx = np.where(frame_fd['cur_3d_idx'] == lid)[0]
                if len(obs_idx) > 0:
                    # Found first observation of this landmark
                    z_cam = frame_fd['cur_3d'][obs_idx[0]]
                    # Use the pose at that frame to transform to world
                    frame_pose = frame_fd['pose_frontend']
                    z_cam_homo = np.append(z_cam, 1.0)
                    lm_init = (frame_pose @ z_cam_homo)[:3]
                    lm['initial'] = lm_init
                    break
    
    # Separate landmarks with and without initial positions
    landmarks_with_init = [lm for lm in all_landmarks if lm['initial'] is not None]
    landmarks_no_init = [lm for lm in all_landmarks if lm['initial'] is None]
    
    print(f"  {len(landmarks_with_init)} landmarks with initial positions")
    print(f"  {len(landmarks_no_init)} landmarks without initial positions (newly added)")
    
    # ===== GET ALL KEYFRAME POSES =====
    all_kf_poses_opt = []
    all_kf_poses_init = []
    for kf_idx in range(global_opt.get_num_poses()):
        kf_sym = gtsam.symbol('x', kf_idx)
        if global_opt._values.exists(kf_sym):
            kf_pose_opt_gtsam = global_opt._values.atPose3(kf_sym)
            kf_pose_opt = inverse_SE3(kf_pose_opt_gtsam.matrix())
            all_kf_poses_opt.append(kf_pose_opt)
            # Get initial pose for this keyframe
            if kf_idx < len(KEYFRAME_INDICES):
                traj_idx_kf = KEYFRAME_INDICES[kf_idx]
                kf_pose_init = traj_local[traj_idx_kf] if traj_idx_kf < len(traj_local) else None
                if kf_pose_init is None:
                    kf_pose_init = kf_pose_opt  # Fallback
                all_kf_poses_init.append(kf_pose_init)
            else:
                all_kf_poses_init.append(kf_pose_opt)
    
    # ===== VISUALIZATION =====
    fig = plt.figure(figsize=(20, 14))
    
    # Main 3D plot
    ax = fig.add_subplot(2, 3, (1, 4), projection='3d')
    
    # Draw all keyframe poses (initial)
    for i, kf_pose_init in enumerate(all_kf_poses_init):
        kf_origin = kf_pose_init[:3, 3]
        if i == kf_idx_in_graph:
            # Highlight the selected keyframe
            ax.scatter(*kf_origin, c='red', marker='^', s=300, alpha=0.8, 
                      edgecolors='darkred', linewidths=3, label='Selected KF (Init)' if i == 0 else '')
        else:
            ax.scatter(*kf_origin, c='lightcoral', marker='^', s=100, alpha=0.4, 
                      edgecolors='red', linewidths=1)
    
    # Draw all keyframe poses (optimized)
    for i, kf_pose_opt in enumerate(all_kf_poses_opt):
        kf_origin = kf_pose_opt[:3, 3]
        if i == kf_idx_in_graph:
            # Highlight the selected keyframe
            ax.scatter(*kf_origin, c='magenta', marker='^', s=300, alpha=0.9, 
                      edgecolors='darkmagenta', linewidths=3, label='Selected KF (Opt)' if i == 0 else '')
            # Draw coordinate axes for selected keyframe
            axes_scale = 0.08
            opt_origin = opt_pose[:3, 3]
            opt_x = opt_origin + opt_pose[:3, 0] * axes_scale
            opt_y = opt_origin + opt_pose[:3, 1] * axes_scale
            opt_z = opt_origin + opt_pose[:3, 2] * axes_scale
            ax.plot([opt_origin[0], opt_x[0]], [opt_origin[1], opt_x[1]], [opt_origin[2], opt_x[2]], 
                   'r-', linewidth=3, alpha=0.8)
            ax.plot([opt_origin[0], opt_y[0]], [opt_origin[1], opt_y[1]], [opt_origin[2], opt_y[2]], 
                   'g-', linewidth=3, alpha=0.8)
            ax.plot([opt_origin[0], opt_z[0]], [opt_origin[1], opt_z[1]], [opt_origin[2], opt_z[2]], 
                   'b-', linewidth=3, alpha=0.8)
        else:
            ax.scatter(*kf_origin, c='plum', marker='^', s=100, alpha=0.4, 
                      edgecolors='magenta', linewidths=1)
    
    # Draw initial landmarks (only those with initial positions)
    if len(landmarks_with_init) > 0:
        initial_lms = np.array([lm['initial'] for lm in landmarks_with_init])
        # Color by distance moved
        landmark_diffs = np.array([np.linalg.norm(lm['optimized'] - lm['initial']) 
                                   for lm in landmarks_with_init])
        scatter1 = ax.scatter(initial_lms[:, 0], initial_lms[:, 1], initial_lms[:, 2], 
                            c=landmark_diffs, cmap='Oranges', marker='o', s=60, alpha=0.7, 
                            edgecolors='darkorange', linewidths=1.5, label='Initial Landmarks',
                            vmin=0, vmax=max(landmark_diffs) if len(landmark_diffs) > 0 else 1)
    
    # Draw optimized landmarks
    if len(all_landmarks) > 0:
        optimized_lms = np.array([lm['optimized'] for lm in all_landmarks])
        if len(landmarks_with_init) > 0:
            # Color optimized landmarks by how much they moved
            landmark_diffs_all = np.array([np.linalg.norm(lm['optimized'] - lm['initial']) 
                                          if lm['initial'] is not None else 0 
                                          for lm in all_landmarks])
            scatter2 = ax.scatter(optimized_lms[:, 0], optimized_lms[:, 1], optimized_lms[:, 2], 
                                c=landmark_diffs_all, cmap='viridis', marker='X', s=80, alpha=0.9, 
                                edgecolors='darkblue', linewidths=2, label='Optimized Landmarks',
                                vmin=0, vmax=max(landmark_diffs_all) if len(landmark_diffs_all) > 0 else 1)
        else:
            ax.scatter(optimized_lms[:, 0], optimized_lms[:, 1], optimized_lms[:, 2], 
                      c='cyan', marker='X', s=80, alpha=0.9, edgecolors='darkblue', 
                      linewidths=2, label='Optimized Landmarks')
    
    # Draw arrows showing landmark movement (only for landmarks with initial positions)
    if len(landmarks_with_init) > 0:
        initial_lms = np.array([lm['initial'] for lm in landmarks_with_init])
        optimized_lms_matched = np.array([lm['optimized'] for lm in landmarks_with_init])
        
        # Use quiver for better arrow visualization
        for i in range(len(landmarks_with_init)):
            start = initial_lms[i]
            end = optimized_lms_matched[i]
            diff = end - start
            dist = np.linalg.norm(diff)
            
            if dist > 1e-6:  # Only draw if there's significant movement
                # Draw arrow with color based on distance
                color_intensity = min(1.0, dist / (max(landmark_diffs) + 1e-6))
                ax.quiver(start[0], start[1], start[2], 
                         diff[0], diff[1], diff[2],
                         color=plt.cm.Reds(color_intensity), 
                         arrow_length_ratio=0.3, alpha=0.6, linewidth=1.5, 
                         length=dist)
    
    ax.set_xlabel('X (m)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Y (m)', fontsize=11, fontweight='bold')
    ax.set_zlabel('Z (m)', fontsize=11, fontweight='bold')
    ax.set_title(f'Keyframe {kf_frame_id}: All Landmarks & Camera Poses\n(Arrows show optimization movement)', 
                fontsize=12, fontweight='bold', pad=15)
    ax.legend(loc='upper left', fontsize=9, framealpha=0.9)
    ax.grid(True, alpha=0.3)
    
    # Add colorbar for landmark movement
    if len(landmarks_with_init) > 0 and len(landmark_diffs) > 0:
        cbar = plt.colorbar(scatter2, ax=ax, shrink=0.6, pad=0.1)
        cbar.set_label('Landmark Movement (m)', fontsize=9, fontweight='bold')
    
    # 2D Projection: X-Y view
    ax1 = fig.add_subplot(2, 3, 2)
    # All keyframes
    for i, (kf_init, kf_opt) in enumerate(zip(all_kf_poses_init, all_kf_poses_opt)):
        if i == kf_idx_in_graph:
            ax1.scatter(kf_init[:3, 3][0], kf_init[:3, 3][1], c='red', marker='^', s=200, 
                       alpha=0.8, edgecolors='darkred', linewidths=2, label='Selected KF (Init)' if i == 0 else '')
            ax1.scatter(kf_opt[:3, 3][0], kf_opt[:3, 3][1], c='magenta', marker='^', s=200, 
                       alpha=0.9, edgecolors='darkmagenta', linewidths=2, label='Selected KF (Opt)' if i == 0 else '')
        else:
            ax1.scatter(kf_init[:3, 3][0], kf_init[:3, 3][1], c='lightcoral', marker='^', s=80, alpha=0.4)
            ax1.scatter(kf_opt[:3, 3][0], kf_opt[:3, 3][1], c='plum', marker='^', s=80, alpha=0.4)
    
    # Landmarks with arrows
    if len(landmarks_with_init) > 0:
        initial_lms = np.array([lm['initial'] for lm in landmarks_with_init])
        optimized_lms_matched = np.array([lm['optimized'] for lm in landmarks_with_init])
        ax1.scatter(initial_lms[:, 0], initial_lms[:, 1], c='orange', marker='o', s=40, 
                   alpha=0.6, edgecolors='darkorange', linewidths=1, label='Initial Landmarks')
        ax1.scatter(optimized_lms_matched[:, 0], optimized_lms_matched[:, 1], c='cyan', marker='X', s=60, 
                   alpha=0.8, edgecolors='darkblue', linewidths=1.5, label='Optimized Landmarks')
        
        # Draw arrows
        for i in range(len(landmarks_with_init)):
            start = initial_lms[i]
            end = optimized_lms_matched[i]
            diff = end - start
            dist = np.linalg.norm(diff)
            if dist > 1e-6:
                ax1.arrow(start[0], start[1], diff[0], diff[1], 
                         head_width=0.005, head_length=0.005, fc='red', ec='red', 
                         alpha=0.5, linewidth=1, length_includes_head=True)
    
    # Landmarks without initial positions
    if len(landmarks_no_init) > 0:
        no_init_lms = np.array([lm['optimized'] for lm in landmarks_no_init])
        ax1.scatter(no_init_lms[:, 0], no_init_lms[:, 1], c='gray', marker='X', s=40, 
                   alpha=0.5, edgecolors='black', linewidths=1, label='New Landmarks')
    
    ax1.set_xlabel('X (m)', fontsize=10, fontweight='bold')
    ax1.set_ylabel('Y (m)', fontsize=10, fontweight='bold')
    ax1.set_title('X-Y Projection', fontsize=11, fontweight='bold')
    ax1.legend(fontsize=8, framealpha=0.9)
    ax1.axis('equal')
    ax1.grid(True, alpha=0.3)
    
    # 2D Projection: X-Z view
    ax2 = fig.add_subplot(2, 3, 3)
    for i, (kf_init, kf_opt) in enumerate(zip(all_kf_poses_init, all_kf_poses_opt)):
        if i == kf_idx_in_graph:
            ax2.scatter(kf_init[:3, 3][0], kf_init[:3, 3][2], c='red', marker='^', s=200, 
                       alpha=0.8, edgecolors='darkred', linewidths=2)
            ax2.scatter(kf_opt[:3, 3][0], kf_opt[:3, 3][2], c='magenta', marker='^', s=200, 
                       alpha=0.9, edgecolors='darkmagenta', linewidths=2)
        else:
            ax2.scatter(kf_init[:3, 3][0], kf_init[:3, 3][2], c='lightcoral', marker='^', s=80, alpha=0.4)
            ax2.scatter(kf_opt[:3, 3][0], kf_opt[:3, 3][2], c='plum', marker='^', s=80, alpha=0.4)
    
    if len(landmarks_with_init) > 0:
        initial_lms = np.array([lm['initial'] for lm in landmarks_with_init])
        optimized_lms_matched = np.array([lm['optimized'] for lm in landmarks_with_init])
        ax2.scatter(initial_lms[:, 0], initial_lms[:, 2], c='orange', marker='o', s=40, 
                   alpha=0.6, edgecolors='darkorange', linewidths=1)
        ax2.scatter(optimized_lms_matched[:, 0], optimized_lms_matched[:, 2], c='cyan', marker='X', s=60, 
                   alpha=0.8, edgecolors='darkblue', linewidths=1.5)
        for i in range(len(landmarks_with_init)):
            start = initial_lms[i]
            end = optimized_lms_matched[i]
            diff = end - start
            dist = np.linalg.norm(diff)
            if dist > 1e-6:
                ax2.arrow(start[0], start[2], diff[0], diff[2], 
                         head_width=0.005, head_length=0.005, fc='red', ec='red', 
                         alpha=0.5, linewidth=1, length_includes_head=True)
    
    if len(landmarks_no_init) > 0:
        no_init_lms = np.array([lm['optimized'] for lm in landmarks_no_init])
        ax2.scatter(no_init_lms[:, 0], no_init_lms[:, 2], c='gray', marker='X', s=40, 
                   alpha=0.5, edgecolors='black', linewidths=1)
    
    ax2.set_xlabel('X (m)', fontsize=10, fontweight='bold')
    ax2.set_ylabel('Z (m)', fontsize=10, fontweight='bold')
    ax2.set_title('X-Z Projection', fontsize=11, fontweight='bold')
    ax2.axis('equal')
    ax2.grid(True, alpha=0.3)
    
    # Landmark movement histogram
    ax3 = fig.add_subplot(2, 3, 5)
    if len(landmarks_with_init) > 0:
        landmark_diffs = np.array([np.linalg.norm(lm['optimized'] - lm['initial']) 
                                  for lm in landmarks_with_init])
        ax3.hist(landmark_diffs * 1000, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
        ax3.axvline(np.mean(landmark_diffs) * 1000, color='red', linestyle='--', linewidth=2, 
                   label=f'Mean: {np.mean(landmark_diffs)*1000:.2f} mm')
        ax3.set_xlabel('Landmark Movement (mm)', fontsize=10, fontweight='bold')
        ax3.set_ylabel('Count', fontsize=10, fontweight='bold')
        ax3.set_title('Landmark Movement Distribution', fontsize=11, fontweight='bold')
        ax3.legend(fontsize=9)
        ax3.grid(True, alpha=0.3, axis='y')
    
    # Statistics panel
    ax4 = fig.add_subplot(2, 3, 6)
    ax4.axis('off')
    
    pose_translation_diff = np.linalg.norm(opt_pose[:3, 3] - init_pose[:3, 3])
    
    stats_text = f"Keyframe {kf_frame_id} Statistics\n"
    stats_text += "=" * 50 + "\n\n"
    stats_text += f"Total Landmarks in Graph: {len(all_landmarks)}\n"
    stats_text += f"  - With Initial Positions: {len(landmarks_with_init)}\n"
    stats_text += f"  - Newly Added: {len(landmarks_no_init)}\n"
    stats_text += f"Total Keyframes: {len(all_kf_poses_opt)}\n\n"
    
    stats_text += "Selected Camera Pose:\n"
    stats_text += f"  Initial: [{init_pose[:3, 3][0]:.4f}, {init_pose[:3, 3][1]:.4f}, {init_pose[:3, 3][2]:.4f}]\n"
    stats_text += f"  Optimized: [{opt_pose[:3, 3][0]:.4f}, {opt_pose[:3, 3][1]:.4f}, {opt_pose[:3, 3][2]:.4f}]\n"
    stats_text += f"  Translation Change: {pose_translation_diff*1000:.2f} mm\n\n"
    
    if len(landmarks_with_init) > 0:
        landmark_diffs = np.array([np.linalg.norm(lm['optimized'] - lm['initial']) 
                                  for lm in landmarks_with_init])
        stats_text += "Landmark Movement:\n"
        stats_text += f"  Mean: {np.mean(landmark_diffs)*1000:.2f} mm\n"
        stats_text += f"  Median: {np.median(landmark_diffs)*1000:.2f} mm\n"
        stats_text += f"  Max: {np.max(landmark_diffs)*1000:.2f} mm\n"
        stats_text += f"  Min: {np.min(landmark_diffs)*1000:.2f} mm\n"
        stats_text += f"  Std Dev: {np.std(landmark_diffs)*1000:.2f} mm\n"
    
    ax4.text(0.05, 0.5, stats_text, fontsize=10, family='monospace', 
            verticalalignment='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return {
        'frame_id': kf_frame_id,
        'initial_pose': init_pose,
        'optimized_pose': opt_pose,
        'all_landmarks': all_landmarks,
        'landmarks_with_init': landmarks_with_init,
        'num_landmarks': len(all_landmarks)
    }

# Example: Visualize first keyframe
if len(KEYFRAME_FRAME_IDS) > 0:
    print(f"\nExample: Visualizing first keyframe (Frame ID {KEYFRAME_FRAME_IDS[0]})")
    print("To visualize a different keyframe, call: visualize_keyframe(frame_id)")
    print(f"Available frame IDs: {KEYFRAME_FRAME_IDS}")


## Single Frame Debugging
Inspect a specific frame to see why optimization might be failing or behaving oddly.

In [ ]:
# visualize_keyframe(212)

In [ ]:
debug_frame_idx = 176

# Retrieve data
fd = get_frame_data(meta_data, debug_frame_idx)
print(f"--- Frame {debug_frame_idx} (Frame ID: {fd['frame_id']}) ---")
print(f"Num Feature Points: {len(fd['cur_3d'])}")
print(f"Num Inliers: {np.sum(fd['inliers'])}")
print(f"Is Keyframe: {fd['is_keyframe']}")

# Check local optimizer graph for this frame
local_opt = local_optimizer._get_optimizer(0)
frame_sym = gtsam.symbol('x', fd['frame_id'])

print("\n--- Local Optimizer State ---")
if hasattr(local_opt, '_values') and local_opt._values.exists(frame_sym):
    pose_est_gtsam = local_opt._values.atPose3(frame_sym)
    print("Optimized Pose (in Local Graph):\n", pose_est_gtsam)
    
    # Inspect Factors connected to this variable
    print("\n--- Connected Factors (Local) ---")
    graph = local_opt._graph
    factor_count = 0
    for i in range(graph.size()):
        factor = graph.at(i)
        keys = factor.keys()
        if frame_sym in keys:
            factor_count += 1
            if factor_count <= 5:  # Show first 5 factors
                print(f"Factor {i} Type: {type(factor).__name__}")
                print(f"  Keys: {[gtsam.DefaultKeyFormatter(k) for k in keys]}")
                print(f"  Error: {factor.error(local_opt._values):.4f}")
    if factor_count > 5:
        print(f"... and {factor_count - 5} more factors")
else:
    print("Frame not found in local graph!")

# Check global optimizer graph (if this is a keyframe)
if fd['is_keyframe']:
    print("\n--- Global Optimizer State ---")
    global_opt = kf_graph._get_optimizer(0)
    # Find the keyframe index for this frame
    # We'd need to track this, but for now just check if any frames exist
    if hasattr(global_opt, '_values'):
        num_kfs = global_opt.get_num_poses()
        print(f"Number of keyframes in global graph: {num_kfs}")
        if num_kfs > 0:
            print("Global graph contains keyframes (use frame inspection to see details)")

In [ ]:
# 3D Visualization of Points for specific frame (Interactive/Rotatable)
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Points in Camera Frame (Observations)
pts_obs = fd['cur_3d']
if len(pts_obs) > 0:
    ax.scatter(pts_obs[:,0], pts_obs[:,1], pts_obs[:,2], c='b', s=20, alpha=0.5, label='Observed (Cam Frame)')

# Estimated Points from Local Optimizer
local_opt = local_optimizer._get_optimizer(0)
frame_sym = gtsam.symbol('x', fd['frame_id'])

est_pts_local = []
valid_indices_local = []

if hasattr(local_opt, '_values') and local_opt._values.exists(frame_sym):
    pose_est_gtsam = local_opt._values.atPose3(frame_sym)
    
    for i, track_id in enumerate(fd['cur_3d_idx']):
        lm_sym = gtsam.symbol('l', int(track_id))
        if local_opt._values.exists(lm_sym):
            # L_world (or object frame, depending on optimizer convention)
            p_w = local_opt._values.atPoint3(lm_sym)
            # Transform to Camera: p_c = pose_est.transformTo(p_w)
            # Note: pose_est is stored as inverse(pose) in LMGraphOptimizer
            p_c = pose_est_gtsam.transformTo(p_w)
            est_pts_local.append(p_c)
            valid_indices_local.append(i)
            
    est_pts_local = np.array(est_pts_local)
    if len(est_pts_local) > 0:
        ax.scatter(est_pts_local[:,0], est_pts_local[:,1], est_pts_local[:,2], 
                  c='r', marker='x', s=30, label='Estimated (Local Graph)', linewidths=1.5)

# Show GT pose if available
if fd['gt_pose'] is not None:
    gt_pose = fd['gt_pose']
    # Draw coordinate frame for GT pose
    origin = gt_pose[:3, 3]
    ax.scatter(*origin, c='g', marker='^', s=100, label='GT Pose Origin')
    
    # Draw axes
    scale = 0.05
    x_axis = origin + gt_pose[:3, 0] * scale
    y_axis = origin + gt_pose[:3, 1] * scale
    z_axis = origin + gt_pose[:3, 2] * scale
    ax.plot([origin[0], x_axis[0]], [origin[1], x_axis[1]], [origin[2], x_axis[2]], 'r-', linewidth=2)
    ax.plot([origin[0], y_axis[0]], [origin[1], y_axis[1]], [origin[2], y_axis[2]], 'g-', linewidth=2)
    ax.plot([origin[0], z_axis[0]], [origin[1], z_axis[1]], [origin[2], z_axis[2]], 'b-', linewidth=2)

ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_zlabel('Z (m)')
plt.title(f"Frame {debug_frame_idx}: Observed vs Estimated Points (Camera Frame)")
plt.legend()
plt.show()

# Print pose comparison
print(f"\n--- Pose Comparison for Frame {debug_frame_idx} ---")
print(f"Frontend Pose (translation): {fd['pose_frontend'][:3, 3]}")
print(f"Local Optimized Pose (translation): {fd['pose_local'][:3, 3]}")
if fd['gt_pose'] is not None:
    print(f"GT Pose (translation): {fd['gt_pose'][:3, 3]}")
    trans_err = np.linalg.norm(fd['pose_local'][:3, 3] - fd['gt_pose'][:3, 3])
    print(f"Translation Error: {trans_err:.4f} m")

In [ ]:
# Factor graph visualization: pose and landmark connections (similar to notebook 12)
# Here we visualize the factor graph inside the local LM optimizer used by the pipeline.

import gtsam


def plot_factor_graph_local(ax, local_optimizer, num_frames_to_show=None, title="Local Factor Graph", use_frontend_poses=True):
    """Plot pose and landmark graph for the local LM optimizer.

    - Poses: nodes `x_i`
    - Landmarks: nodes `l_j`
    - Edges: pose-pose (BetweenFactorPose3) and pose-landmark (BearingRangeFactor3D)

    Args:
        ax: matplotlib 3D axis.
        local_optimizer: `LocalOptimizer` instance used in this notebook.
        num_frames_to_show: optional cap on how many frames (poses) to visualize.
        title: plot title.
        use_frontend_poses: if True, use `traj_frontend` (if available) for pose positions;
                            otherwise use `traj_local` as a fallback.
    """
    # Underlying LM graph optimizer for object 0
    opt = local_optimizer._get_optimizer(0)
    graph = opt._graph
    values = opt._values

    pose_positions = {}
    landmark_positions = {}

    # Decide how many frames to show
    if num_frames_to_show is None:
        num_frames_to_show = len(traj_frontend) if len(traj_frontend) > 0 else len(traj_local)

    # Poses (x_i)
    for i in range(num_frames_to_show):
        Xi = gtsam.symbol('x', i)
        if values.exists(Xi):
            try:
                # Use frontend or local trajectory as proxy for world pose if available
                if use_frontend_poses and len(traj_frontend) > i:
                    p = traj_frontend[i][:3, 3]
                elif len(traj_local) > i:
                    p = traj_local[i][:3, 3]
                else:
                    # Fallback: read pose from optimizer values (cam_T_world)
                    pose_c2w = values.atPose3(Xi).matrix()
                    p = pose_c2w[:3, 3]
                pose_positions[Xi] = p
                ax.scatter(*p, c='blue', s=35, alpha=0.85, marker='o')
            except RuntimeError:
                continue

    # Landmarks (l_j)
    # We do not have an explicit list of landmark IDs on LocalOptimizer,
    # so we scan the graph factors for landmark keys.
    landmark_ids_seen = set()
    for k in range(graph.size()):
        f = graph.at(k)
        for key in f.keys():
            if gtsam.symbolChr(key) == ord('l'):
                lid = gtsam.symbolIndex(key)
                if lid in landmark_ids_seen:
                    continue
                landmark_ids_seen.add(lid)
                Lj = gtsam.symbol('l', int(lid))
                if values.exists(Lj):
                    try:
                        p = values.atPoint3(Lj)
                        landmark_positions[Lj] = p
                        ax.scatter(*p, c='red', s=25, alpha=0.85, marker='^')
                    except RuntimeError:
                        continue

    # Draw factor edges
    num_factors = graph.size()
    between_count = 0
    bearing_count = 0

    for k in range(num_factors):
        f = graph.at(k)
        # Pose-pose constraints
        if isinstance(f, gtsam.BetweenFactorPose3):
            keys = f.keys()
            if len(keys) == 2:
                k1, k2 = keys[0], keys[1]
                if k1 in pose_positions and k2 in pose_positions:
                    p1 = pose_positions[k1]
                    p2 = pose_positions[k2]
                    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]],
                            color='blue', alpha=0.4, linewidth=1.5)
                    between_count += 1
        # Pose-landmark bearing-range constraints
        elif isinstance(f, gtsam.BearingRangeFactor3D):
            keys = f.keys()
            if len(keys) == 2:
                k_pose, k_land = keys[0], keys[1]
                if k_pose in pose_positions and k_land in landmark_positions:
                    p1 = pose_positions[k_pose]
                    p2 = landmark_positions[k_land]
                    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]],
                            color='green', alpha=0.3, linewidth=0.8)
                    bearing_count += 1

    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_zlabel('Z (m)')
    ax.set_title(f"{title}\n({len(pose_positions)} poses, {len(landmark_positions)} landmarks, "
                 f"{between_count} between factors, {bearing_count} bearing-range factors)",
                 fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.view_init(elev=30, azim=-60)


# Example usage: visualize the local factor graph after the main optimization loop above has run.
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
plot_factor_graph_local(ax, local_optimizer, title="Local LM Factor Graph (pipeline replay)")
plt.tight_layout()
plt.show()


In [ ]:
# Factor graph visualization for GLOBAL keyframe optimizer (similar to notebook 12)
# Visualizes the keyframe graph with pose-pose and pose-landmark connections

def plot_factor_graph_global(ax, kf_graph, keyframe_indices, traj_global, title="Global Keyframe Factor Graph"):
    """Plot pose and landmark graph for the global keyframe optimizer.

    - Keyframe Poses: nodes `x_i` where i is kf_idx (0, 1, 2, ...)
    - Landmarks: nodes `l_j`
    - Edges: pose-pose (BetweenFactorPose3) and pose-landmark (BearingRangeFactor3D)

    Args:
        ax: matplotlib 3D axis.
        kf_graph: `KeyFrameGraph` instance used in this notebook.
        keyframe_indices: list of trajectory indices for each keyframe (maps kf_idx -> traj_idx).
        traj_global: global trajectory array (contains optimized keyframe poses).
        title: plot title.
    """
    # Get underlying optimizer for object 0
    opt = kf_graph._get_optimizer(0)
    graph = opt._graph
    values = opt._values

    pose_positions = {}
    landmark_positions = {}

    # Extract keyframe poses (indexed by kf_idx, not frame_id)
    num_keyframes = len(keyframe_indices) if keyframe_indices else 0
    
    for kf_idx in range(num_keyframes):
        Xi = gtsam.symbol('x', kf_idx)
        if values.exists(Xi):
            try:
                # Read pose directly from optimizer values (this is the authoritative state)
                pose_c2w = values.atPose3(Xi).matrix()
                p = pose_c2w[:3, 3]

                pose_positions[Xi] = p
                ax.scatter(*p, c='blue', s=50, alpha=0.9, marker='o', 
                          edgecolors='darkblue', linewidths=1.5)
            except RuntimeError:
                continue

    # Landmarks (l_j) - scan graph factors for landmark keys
    landmark_ids_seen = set()
    for k in range(graph.size()):
        f = graph.at(k)
        for key in f.keys():
            if gtsam.symbolChr(key) == ord('l'):
                lid = gtsam.symbolIndex(key)
                if lid in landmark_ids_seen:
                    continue
                landmark_ids_seen.add(lid)
                Lj = gtsam.symbol('l', int(lid))
                if values.exists(Lj):
                    try:
                        p = values.atPoint3(Lj)
                        landmark_positions[Lj] = p
                        ax.scatter(*p, c='red', s=30, alpha=0.85, marker='^',
                                  edgecolors='darkred', linewidths=1.5)
                    except RuntimeError:
                        continue
    
    # Draw factor edges
    num_factors = graph.size()
    between_count = 0
    bearing_count = 0

    for k in range(num_factors):
        f = graph.at(k)
        # Pose-pose constraints (between keyframes)
        if isinstance(f, gtsam.BetweenFactorPose3):
            keys = f.keys()
            if len(keys) == 2:
                k1, k2 = keys[0], keys[1]
                if k1 in pose_positions and k2 in pose_positions:
                    p1 = pose_positions[k1]
                    p2 = pose_positions[k2]
                    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]],
                            color='blue', alpha=0.5, linewidth=2.0)
                    between_count += 1
        # Pose-landmark bearing-range constraints
        elif isinstance(f, gtsam.BearingRangeFactor3D):
            keys = f.keys()
            if len(keys) == 2:
                k_pose, k_land = keys[0], keys[1]
                if k_pose in pose_positions and k_land in landmark_positions:
                    p1 = pose_positions[k_pose]
                    p2 = landmark_positions[k_land]
                    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]],
                            color='green', alpha=0.35, linewidth=1.0)
                    bearing_count += 1

    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_zlabel('Z (m)')
    ax.set_title(f"{title}\n({len(pose_positions)} keyframes, {len(landmark_positions)} landmarks, "
                 f"{between_count} between factors, {bearing_count} bearing-range factors)",
                 fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.view_init(elev=30, azim=-60)


# Visualize the global keyframe factor graph
if len(keyframe_indices) > 0:
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    plot_factor_graph_global(ax, kf_graph, keyframe_indices, traj_global, 
                            title="Global Keyframe Factor Graph (pipeline replay)")
    plt.tight_layout()
    plt.show()
else:
    print("No keyframes found. Global graph visualization requires at least one keyframe.")


In [ ]:
# kf_graph._get_optimizer(0).inserted_landmark_ids


In [ ]:
# Extract Ground Truth Landmarks and Estimated Landmarks for Comparison
# GT landmarks: Transform keypoints from each keyframe back to world using GT poses
# Local landmarks: Extract from local optimizer values
# Global landmarks: Extract from global optimizer values

def extract_gt_landmarks_from_keyframes(meta_data, keyframe_frame_ids, keyframe_indices, start_frame, end_frame, reader=None):
    """Extract ground truth landmarks by transforming keypoints from keyframes to world using GT poses."""
    gt_landmarks_dict = {}  # track_id -> world position
    
    num_keyframes_processed = 0
    num_keyframes_with_gt = 0
    num_keyframes_with_keypoints = 0
    total_keypoints = 0
    
    for kf_idx, frame_idx_in_traj in enumerate(keyframe_indices):
        # Find the actual frame index in metadata
        frame_idx_in_meta = start_frame + frame_idx_in_traj
        if frame_idx_in_meta >= end_frame:
            continue
        
        num_keyframes_processed += 1
        fd = get_frame_data(meta_data, frame_idx_in_meta, reader=reader)
        
        # Get GT pose for this keyframe
        if fd['gt_pose'] is None:
            continue
        
        num_keyframes_with_gt += 1
        gt_pose = fd['gt_pose']
        
        # Get keypoints in camera frame (cur_3d) and their track IDs (cur_3d_idx)
        cur_3d = fd['cur_3d']  # Points in camera frame
        cur_3d_idx = fd['cur_3d_idx']  # Track IDs
        
        if len(cur_3d) == 0:
            continue
        
        num_keyframes_with_keypoints += 1
        total_keypoints += len(cur_3d)
        
        # Transform each keypoint to object frame using GT pose
        # gt_pose is T_obj_cam (object pose in camera frame, same as T_o2c)
        # To transform from camera frame to object frame: use inverse_SE3(gt_pose)
        # This matches the pattern in key_frame_manager.py: xyz_obj = transform_pts(inverse_SE3(T_o2c), xyz_cam)
        for i in range(len(cur_3d)):
            track_id = int(cur_3d_idx[i])
            pt_cam = cur_3d[i]
            
            # Transform from camera frame to object frame
            # gt_pose is T_obj_cam (object to camera transformation)
            # inverse_SE3(gt_pose) transforms from camera to object frame
            pt_obj = transform_pts(inverse_SE3(gt_pose), pt_cam.reshape(1, -1))[0]
            
            # Store or update (use first observation or average)
            if track_id not in gt_landmarks_dict:
                gt_landmarks_dict[track_id] = pt_obj
            else:
                # Average if seen multiple times (or use first, or latest)
                # For now, use latest observation
                gt_landmarks_dict[track_id] = pt_obj
    
    return gt_landmarks_dict


def extract_landmarks_from_optimizer(optimizer, obj_id=0):
    """Extract landmarks from optimizer values."""
    opt = optimizer._get_optimizer(obj_id)
    values = opt._values
    
    landmarks_dict = {}
    
    # Get landmark IDs from optimizer if available
    if hasattr(opt, 'inserted_landmark_ids'):
        landmark_ids = opt.inserted_landmark_ids
    else:
        # Scan graph for landmark keys
        landmark_ids = set()
        graph = opt._graph
        for k in range(graph.size()):
            f = graph.at(k)
            for key in f.keys():
                if gtsam.symbolChr(key) == ord('l'):
                    lid = gtsam.symbolIndex(key)
                    landmark_ids.add(int(lid))
        landmark_ids = sorted(landmark_ids)
    
    # Extract landmark positions
    for lid in landmark_ids:
        Lj = gtsam.symbol('l', int(lid))
        if values.exists(Lj):
            try:
                p = values.atPoint3(Lj)
                landmarks_dict[int(lid)] = np.array(p)
            except RuntimeError:
                continue
    
    return landmarks_dict


# Extract GT landmarks
print("Extracting ground truth landmarks from keyframes...")
gt_landmarks_dict = extract_gt_landmarks_from_keyframes(
    meta_data, keyframe_frame_ids, keyframe_indices, start_frame, end_frame, reader=gt_reader
)
print(f"Found {len(gt_landmarks_dict)} GT landmarks")

# Align GT landmarks using the same alignment transform as GT poses
# The alignment transform T_align = pred_pose @ inv(gt_pose) transforms GT object frame to predicted object frame
# When applied to GT poses: gt_pose_aligned = gt_pose @ T_align
# For points in object frame, we need to apply the inverse: pt_aligned = inv(T_align) @ pt_gt
# This is because poses transform points as T @ pt, but T_align was computed for pose composition
try:
    # Try to access T_align from the alignment code (from Cell 4)
    T_align = None
    if 'T_align' in locals():
        T_align = locals()['T_align']
    elif 'T_align' in globals():
        T_align = globals()['T_align']
    else:
        # Recompute alignment transform from first valid frame
        first_valid_idx = None
        first_valid_gt = None
        for i in range(start_frame, end_frame):
            fd = get_frame_data(meta_data, i, reader=gt_reader)
            if fd['gt_pose'] is not None and (i - start_frame) < len(traj_global):
                first_valid_idx = i
                first_valid_gt = fd['gt_pose'].copy()
                break
        
        if first_valid_idx is not None and first_valid_gt is not None:
            pred_pose_first = traj_global[first_valid_idx - start_frame]
            T_align = pred_pose_first @ inverse_SE3(first_valid_gt)
            print(f"\nRecomputed alignment transform for landmarks (from frame {first_valid_idx})")
        else:
            print("\nNo valid GT frames found for landmark alignment")
    
    # Apply alignment to GT landmarks if T_align exists
    if T_align is not None:
        print(f"Applying alignment transform to {len(gt_landmarks_dict)} GT landmarks...")
        # For poses: gt_pose_aligned = gt_pose @ T_align
        # For points: we need inverse_SE3(T_align) because:
        # - T_align transforms GT object frame to predicted object frame when composed with poses
        # - For points, we need the inverse to transform from GT object frame to predicted object frame
        T_align_points = inverse_SE3(T_align)
        for track_id in gt_landmarks_dict:
            pt_gt = gt_landmarks_dict[track_id]
            pt_aligned = transform_pts(T_align_points, pt_gt.reshape(1, -1))[0]
            gt_landmarks_dict[track_id] = pt_aligned
        print("GT landmarks aligned successfully")
    else:
        print("No alignment transform available. GT landmarks not aligned.")
except Exception as e:
    print(f"Warning: Could not align GT landmarks: {e}")
    import traceback
    traceback.print_exc()
    print("Continuing without alignment...")

# Extract local estimated landmarks
print("Extracting local estimated landmarks...")
local_landmarks_dict = extract_landmarks_from_optimizer(local_optimizer, obj_id=0)
print(f"Found {len(local_landmarks_dict)} local estimated landmarks")

# Extract global estimated landmarks
print("Extracting global estimated landmarks...")
global_landmarks_dict = extract_landmarks_from_optimizer(kf_graph, obj_id=0)
print(f"Found {len(global_landmarks_dict)} global estimated landmarks")

# Find common track IDs for comparison
common_ids_local = set(gt_landmarks_dict.keys()) & set(local_landmarks_dict.keys())
common_ids_global = set(gt_landmarks_dict.keys()) & set(global_landmarks_dict.keys())

print(f"\nCommon landmarks (GT vs Local): {len(common_ids_local)}")
print(f"Common landmarks (GT vs Global): {len(common_ids_global)}")

# Prepare arrays for visualization
def prepare_landmark_arrays(gt_dict, est_dict, common_ids):
    """Prepare matched arrays of GT and estimated landmarks."""
    common_ids_sorted = sorted(common_ids)
    gt_array = np.array([gt_dict[lid] for lid in common_ids_sorted])
    est_array = np.array([est_dict[lid] for lid in common_ids_sorted])
    return gt_array, est_array, np.array(common_ids_sorted)

gt_local, est_local, ids_local = prepare_landmark_arrays(
    gt_landmarks_dict, local_landmarks_dict, common_ids_local
)

gt_global, est_global, ids_global = prepare_landmark_arrays(
    gt_landmarks_dict, global_landmarks_dict, common_ids_global
)

print(f"\nPrepared {len(ids_local)} matched landmarks for local comparison")
print(f"Prepared {len(ids_global)} matched landmarks for global comparison")


In [ ]:
# 3D Visualization: GT Landmarks vs Estimated Landmarks
# Two plots: GT vs Local, and GT vs Global

fig = plt.figure(figsize=(16, 6))

# GT vs Local
ax1 = fig.add_subplot(121, projection='3d')
if len(gt_local) > 0:
    ax1.scatter(gt_local[:, 0], gt_local[:, 1], gt_local[:, 2], 
               c='red', s=50, alpha=0.7, marker='o', label='Ground Truth', edgecolors='darkred')
    ax1.scatter(est_local[:, 0], est_local[:, 1], est_local[:, 2], 
               c='purple', s=30, alpha=0.7, marker='^', label='Local Estimated', edgecolors='indigo')
    # Draw lines connecting GT to estimated
    for i in range(len(gt_local)):
        ax1.plot([gt_local[i, 0], est_local[i, 0]], 
                [gt_local[i, 1], est_local[i, 1]], 
                [gt_local[i, 2], est_local[i, 2]], 
                'k--', alpha=0.2, linewidth=0.5)
ax1.set_xlabel('X (m)')
ax1.set_ylabel('Y (m)')
ax1.set_zlabel('Z (m)')
ax1.set_title(f'GT vs Local Estimated Landmarks\n({len(gt_local)} landmarks)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# GT vs Global
ax2 = fig.add_subplot(122, projection='3d')
if len(gt_global) > 0:
    ax2.scatter(gt_global[:, 0], gt_global[:, 1], gt_global[:, 2], 
               c='red', s=50, alpha=0.7, marker='o', label='Ground Truth', edgecolors='darkred')
    ax2.scatter(est_global[:, 0], est_global[:, 1], est_global[:, 2], 
               c='purple', s=30, alpha=0.7, marker='^', label='Global Estimated', edgecolors='indigo')
    # Draw lines connecting GT to estimated
    for i in range(len(gt_global)):
        ax2.plot([gt_global[i, 0], est_global[i, 0]], 
                [gt_global[i, 1], est_global[i, 1]], 
                [gt_global[i, 2], est_global[i, 2]], 
                'k--', alpha=0.2, linewidth=0.5)
ax2.set_xlabel('X (m)')
ax2.set_ylabel('Y (m)')
ax2.set_zlabel('Z (m)')
ax2.set_title(f'GT vs Global Estimated Landmarks\n({len(gt_global)} landmarks)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# 2D Projections: GT vs Local Estimated Landmarks (similar to notebook 12)
fig = plt.figure(figsize=(16, 5))

if len(gt_local) > 0:
    # X-Y view
    ax1 = fig.add_subplot(131)
    ax1.scatter(gt_local[:, 0], gt_local[:, 1], c='red', s=50, alpha=0.7, 
               label='Ground Truth', marker='o', edgecolors='darkred')
    ax1.scatter(est_local[:, 0], est_local[:, 1], c='purple', s=30, alpha=0.7, 
               label='Local Estimated', marker='^', edgecolors='indigo')
    # Draw lines connecting GT to estimated
    for i in range(len(gt_local)):
        ax1.plot([gt_local[i, 0], est_local[i, 0]], 
                [gt_local[i, 1], est_local[i, 1]], 
                'k--', alpha=0.3, linewidth=0.5)
    ax1.set_xlabel('X (m)')
    ax1.set_ylabel('Y (m)')
    ax1.set_title('Landmarks (X-Y view)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_aspect('equal')
    
    # X-Z view
    ax2 = fig.add_subplot(132)
    ax2.scatter(gt_local[:, 0], gt_local[:, 2], c='red', s=50, alpha=0.7, 
               label='Ground Truth', marker='o', edgecolors='darkred')
    ax2.scatter(est_local[:, 0], est_local[:, 2], c='purple', s=30, alpha=0.7, 
               label='Local Estimated', marker='^', edgecolors='indigo')
    for i in range(len(gt_local)):
        ax2.plot([gt_local[i, 0], est_local[i, 0]], 
                [gt_local[i, 2], est_local[i, 2]], 
                'k--', alpha=0.3, linewidth=0.5)
    ax2.set_xlabel('X (m)')
    ax2.set_ylabel('Z (m)')
    ax2.set_title('Landmarks (X-Z view)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_aspect('equal')
    
    # Y-Z view
    ax3 = fig.add_subplot(133)
    ax3.scatter(gt_local[:, 1], gt_local[:, 2], c='red', s=50, alpha=0.7, 
               label='Ground Truth', marker='o', edgecolors='darkred')
    ax3.scatter(est_local[:, 1], est_local[:, 2], c='purple', s=30, alpha=0.7, 
               label='Local Estimated', marker='^', edgecolors='indigo')
    for i in range(len(gt_local)):
        ax3.plot([gt_local[i, 1], est_local[i, 1]], 
                [gt_local[i, 2], est_local[i, 2]], 
                'k--', alpha=0.3, linewidth=0.5)
    ax3.set_xlabel('Y (m)')
    ax3.set_ylabel('Z (m)')
    ax3.set_title('Landmarks (Y-Z view)')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    ax3.set_aspect('equal')
else:
    ax1 = fig.add_subplot(131)
    ax1.text(0.5, 0.5, 'No landmarks available', ha='center', va='center', transform=ax1.transAxes)
    ax1.set_title('Landmarks (X-Y view)')

plt.tight_layout()
plt.show()


In [ ]:
# 2D Projections: GT vs Global Estimated Landmarks
fig = plt.figure(figsize=(16, 5))

if len(gt_global) > 0:
    # X-Y view
    ax1 = fig.add_subplot(131)
    ax1.scatter(gt_global[:, 0], gt_global[:, 1], c='red', s=50, alpha=0.7, 
               label='Ground Truth', marker='o', edgecolors='darkred')
    ax1.scatter(est_global[:, 0], est_global[:, 1], c='purple', s=30, alpha=0.7, 
               label='Global Estimated', marker='^', edgecolors='indigo')
    # Draw lines connecting GT to estimated
    for i in range(len(gt_global)):
        ax1.plot([gt_global[i, 0], est_global[i, 0]], 
                [gt_global[i, 1], est_global[i, 1]], 
                'k--', alpha=0.3, linewidth=0.5)
    ax1.set_xlabel('X (m)')
    ax1.set_ylabel('Y (m)')
    ax1.set_title('Landmarks (X-Y view)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_aspect('equal')
    
    # X-Z view
    ax2 = fig.add_subplot(132)
    ax2.scatter(gt_global[:, 0], gt_global[:, 2], c='red', s=50, alpha=0.7, 
               label='Ground Truth', marker='o', edgecolors='darkred')
    ax2.scatter(est_global[:, 0], est_global[:, 2], c='purple', s=30, alpha=0.7, 
               label='Global Estimated', marker='^', edgecolors='indigo')
    for i in range(len(gt_global)):
        ax2.plot([gt_global[i, 0], est_global[i, 0]], 
                [gt_global[i, 2], est_global[i, 2]], 
                'k--', alpha=0.3, linewidth=0.5)
    ax2.set_xlabel('X (m)')
    ax2.set_ylabel('Z (m)')
    ax2.set_title('Landmarks (X-Z view)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_aspect('equal')
    
    # Y-Z view
    ax3 = fig.add_subplot(133)
    ax3.scatter(gt_global[:, 1], gt_global[:, 2], c='red', s=50, alpha=0.7, 
               label='Ground Truth', marker='o', edgecolors='darkred')
    ax3.scatter(est_global[:, 1], est_global[:, 2], c='purple', s=30, alpha=0.7, 
               label='Global Estimated', marker='^', edgecolors='indigo')
    for i in range(len(gt_global)):
        ax3.plot([gt_global[i, 1], est_global[i, 1]], 
                [gt_global[i, 2], est_global[i, 2]], 
                'k--', alpha=0.3, linewidth=0.5)
    ax3.set_xlabel('Y (m)')
    ax3.set_ylabel('Z (m)')
    ax3.set_title('Landmarks (Y-Z view)')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    ax3.set_aspect('equal')
else:
    ax1 = fig.add_subplot(131)
    ax1.text(0.5, 0.5, 'No landmarks available', ha='center', va='center', transform=ax1.transAxes)
    ax1.set_title('Landmarks (X-Y view)')

plt.tight_layout()
plt.show()

# Extract residuals for landmarks from optimizer graphs
def extract_landmark_residuals(optimizer, obj_id=0, landmark_ids=None):
    """Extract residuals for landmarks from the optimizer graph.
    
    Returns a dictionary mapping landmark_id -> list of residuals (one per factor involving this landmark).
    """
    opt = optimizer._get_optimizer(obj_id)
    graph = opt._graph
    values = opt._values
    
    landmark_residuals = {}
    
    if landmark_ids is None:
        # Get all landmark IDs from optimizer if available
        if hasattr(opt, 'inserted_landmark_ids'):
            landmark_ids = opt.inserted_landmark_ids
        else:
            # Scan graph for landmark keys
            landmark_ids = set()
            for k in range(graph.size()):
                f = graph.at(k)
                for key in f.keys():
                    if gtsam.symbolChr(key) == ord('l'):
                        lid = gtsam.symbolIndex(key)
                        landmark_ids.add(int(lid))
            landmark_ids = sorted(landmark_ids)
    
    # Initialize dictionary
    for lid in landmark_ids:
        landmark_residuals[int(lid)] = []
    
    # Scan all factors to find those involving landmarks
    for k in range(graph.size()):
        f = graph.at(k)
        # Check if this is a BearingRangeFactor3D (pose-landmark factor)
        if isinstance(f, gtsam.BearingRangeFactor3D):
            keys = f.keys()
            if len(keys) == 2:
                # Find the landmark key
                for key in keys:
                    if gtsam.symbolChr(key) == ord('l'):
                        lid = int(gtsam.symbolIndex(key))
                        if lid in landmark_residuals:
                            try:
                                # Compute factor error (residual)
                                residual = f.error(values)
                                landmark_residuals[lid].append(residual)
                            except RuntimeError:
                                pass
                        break
    
    # For each landmark, use mean residual if multiple factors, or single residual
    landmark_mean_residuals = {}
    for lid, residuals in landmark_residuals.items():
        if len(residuals) > 0:
            landmark_mean_residuals[lid] = np.mean(residuals)
        else:
            landmark_mean_residuals[lid] = np.nan
    
    return landmark_mean_residuals

# Extract residuals for local and global optimizers
print("Extracting landmark residuals from optimizers...")
local_residuals_dict = extract_landmark_residuals(local_optimizer, obj_id=0, landmark_ids=ids_local.tolist() if len(ids_local) > 0 else None)
global_residuals_dict = extract_landmark_residuals(kf_graph, obj_id=0, landmark_ids=ids_global.tolist() if len(ids_global) > 0 else None)

print(f"Found residuals for {len([k for k, v in local_residuals_dict.items() if not np.isnan(v)])} local landmarks")
print(f"Found residuals for {len([k for k, v in global_residuals_dict.items() if not np.isnan(v)])} global landmarks")

# Compute errors if not already computed
try:
    _ = local_errors_3d
    _ = global_errors_3d
    _ = local_errors_x
    _ = local_errors_y
    _ = local_errors_z
    _ = global_errors_x
    _ = global_errors_y
    _ = global_errors_z
except NameError:
    # Compute errors now
    local_errors_3d = np.linalg.norm(gt_local - est_local, axis=1) if len(gt_local) > 0 else np.array([])
    global_errors_3d = np.linalg.norm(gt_global - est_global, axis=1) if len(gt_global) > 0 else np.array([])
    local_errors_x = (gt_local[:, 0] - est_local[:, 0]) if len(gt_local) > 0 else np.array([])
    local_errors_y = (gt_local[:, 1] - est_local[:, 1]) if len(gt_local) > 0 else np.array([])
    local_errors_z = (gt_local[:, 2] - est_local[:, 2]) if len(gt_local) > 0 else np.array([])
    global_errors_x = (gt_global[:, 0] - est_global[:, 0]) if len(gt_global) > 0 else np.array([])
    global_errors_y = (gt_global[:, 1] - est_global[:, 1]) if len(gt_global) > 0 else np.array([])
    global_errors_z = (gt_global[:, 2] - est_global[:, 2]) if len(gt_global) > 0 else np.array([])
    print("Computed error arrays for residual matching")

# Match residuals with position errors
local_residuals = []
local_errors_matched = []
for lid in ids_local:
    if lid in local_residuals_dict and not np.isnan(local_residuals_dict[lid]):
        local_residuals.append(local_residuals_dict[lid])
        # Find corresponding error
        idx = np.where(ids_local == lid)[0][0]
        local_errors_matched.append(local_errors_3d[idx])

global_residuals = []
global_errors_matched = []
for lid in ids_global:
    if lid in global_residuals_dict and not np.isnan(global_residuals_dict[lid]):
        global_residuals.append(global_residuals_dict[lid])
        # Find corresponding error
        idx = np.where(ids_global == lid)[0][0]
        global_errors_matched.append(global_errors_3d[idx])

local_residuals = np.array(local_residuals)
local_errors_matched = np.array(local_errors_matched)
global_residuals = np.array(global_residuals)
global_errors_matched = np.array(global_errors_matched)

print(f"\nMatched {len(local_residuals)} local landmarks (error vs residual)")
print(f"Matched {len(global_residuals)} global landmarks (error vs residual)")

# Extract frontend residuals for landmarks
def extract_frontend_residuals(meta_data, keyframe_frame_ids, keyframe_indices, start_frame, end_frame, landmark_ids):
    """Extract frontend residuals for landmarks from metadata.
    
    Only includes residuals when the landmark is an inlier at that frame
    (i.e., when a factor is actually added to the optimizer).
    """
    frontend_residuals_dict = {}
    
    # Initialize
    for lid in landmark_ids:
        frontend_residuals_dict[int(lid)] = []
    
    # Go through keyframes and collect residuals
    for kf_idx, frame_idx_in_traj in enumerate(keyframe_indices):
        frame_idx_in_meta = start_frame + frame_idx_in_traj
        if frame_idx_in_meta >= end_frame:
            continue
            
        fd = get_frame_data(meta_data, frame_idx_in_meta)
        cur_3d_idx = fd['cur_3d_idx']
        residuals = fd['residuals']
        inliers = fd['inliers']
        
        # Match residuals to landmark IDs
        # Only include residuals when the landmark is an inlier (factor is added)
        for i in range(len(cur_3d_idx)):
            track_id = int(cur_3d_idx[i])
            # Check if this landmark is an inlier and has a valid residual
            if (track_id in frontend_residuals_dict and 
                i < len(residuals) and 
                i < len(inliers) and 
                inliers[i]):
                frontend_residuals_dict[track_id].append(residuals[i])
    
    # Average residuals for each landmark
    frontend_mean_residuals = {}
    for lid, residuals in frontend_residuals_dict.items():
        if len(residuals) > 0:
            frontend_mean_residuals[lid] = np.mean(residuals)
        else:
            frontend_mean_residuals[lid] = np.nan
    
    return frontend_mean_residuals

# Extract frontend residuals
print("Extracting frontend residuals for landmarks...")
frontend_residuals_local_dict = extract_frontend_residuals(
    meta_data, keyframe_frame_ids, keyframe_indices, start_frame, end_frame, ids_local.tolist() if len(ids_local) > 0 else []
)
frontend_residuals_global_dict = extract_frontend_residuals(
    meta_data, keyframe_frame_ids, keyframe_indices, start_frame, end_frame, ids_global.tolist() if len(ids_global) > 0 else []
)

# Match frontend residuals with position errors
local_frontend_residuals = []
local_errors_matched_fe = []
for lid in ids_local:
    if lid in frontend_residuals_local_dict and not np.isnan(frontend_residuals_local_dict[lid]):
        local_frontend_residuals.append(frontend_residuals_local_dict[lid])
        idx = np.where(ids_local == lid)[0][0]
        local_errors_matched_fe.append(local_errors_3d[idx])

global_frontend_residuals = []
global_errors_matched_fe = []
for lid in ids_global:
    if lid in frontend_residuals_global_dict and not np.isnan(frontend_residuals_global_dict[lid]):
        global_frontend_residuals.append(frontend_residuals_global_dict[lid])
        idx = np.where(ids_global == lid)[0][0]
        global_errors_matched_fe.append(global_errors_3d[idx])

local_frontend_residuals = np.array(local_frontend_residuals)
local_errors_matched_fe = np.array(local_errors_matched_fe)
global_frontend_residuals = np.array(global_frontend_residuals)
global_errors_matched_fe = np.array(global_errors_matched_fe)

# Also extract X, Y, Z component errors for matched landmarks
local_errors_x_matched = []
local_errors_y_matched = []
local_errors_z_matched = []
for lid in ids_local:
    if lid in local_residuals_dict and not np.isnan(local_residuals_dict[lid]):
        idx = np.where(ids_local == lid)[0][0]
        local_errors_x_matched.append(local_errors_x[idx])
        local_errors_y_matched.append(local_errors_y[idx])
        local_errors_z_matched.append(local_errors_z[idx])

global_errors_x_matched = []
global_errors_y_matched = []
global_errors_z_matched = []
for lid in ids_global:
    if lid in global_residuals_dict and not np.isnan(global_residuals_dict[lid]):
        idx = np.where(ids_global == lid)[0][0]
        global_errors_x_matched.append(global_errors_x[idx])
        global_errors_y_matched.append(global_errors_y[idx])
        global_errors_z_matched.append(global_errors_z[idx])

local_errors_x_matched = np.array(local_errors_x_matched)
local_errors_y_matched = np.array(local_errors_y_matched)
local_errors_z_matched = np.array(local_errors_z_matched)
global_errors_x_matched = np.array(global_errors_x_matched)
global_errors_y_matched = np.array(global_errors_y_matched)
global_errors_z_matched = np.array(global_errors_z_matched)

# Plot 1: Landmark Position Error vs Optimizer Residual (separated by X, Y, Z)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Local optimizer - X error
ax = axes[0, 0]
if len(local_residuals) > 0 and len(local_errors_x_matched) > 0:
    ax.scatter(local_residuals, local_errors_x_matched * 1000, 
              alpha=0.6, s=30, c='blue', edgecolors='darkblue', linewidths=0.5)
    ax.set_xlabel('Factor Residual')
    ax.set_ylabel('X Error vs GT (mm)')
    ax.set_title(f'Local: X Error vs Residual\n({len(local_errors_x_matched)} landmarks)')
    ax.grid(True, alpha=0.3)
    if len(local_residuals) > 1:
        corr = np.corrcoef(local_residuals, local_errors_x_matched)[0, 1]
        ax.text(0.05, 0.95, f'Corr: {corr:.3f}', 
               transform=ax.transAxes, fontsize=9, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Local optimizer - Y error
ax = axes[0, 1]
if len(local_residuals) > 0 and len(local_errors_y_matched) > 0:
    ax.scatter(local_residuals, local_errors_y_matched * 1000, 
              alpha=0.6, s=30, c='blue', edgecolors='darkblue', linewidths=0.5)
    ax.set_xlabel('Factor Residual')
    ax.set_ylabel('Y Error vs GT (mm)')
    ax.set_title(f'Local: Y Error vs Residual\n({len(local_errors_y_matched)} landmarks)')
    ax.grid(True, alpha=0.3)
    if len(local_residuals) > 1:
        corr = np.corrcoef(local_residuals, local_errors_y_matched)[0, 1]
        ax.text(0.05, 0.95, f'Corr: {corr:.3f}', 
               transform=ax.transAxes, fontsize=9, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Local optimizer - Z error
ax = axes[0, 2]
if len(local_residuals) > 0 and len(local_errors_z_matched) > 0:
    ax.scatter(local_residuals, local_errors_z_matched * 1000, 
              alpha=0.6, s=30, c='blue', edgecolors='darkblue', linewidths=0.5)
    ax.set_xlabel('Factor Residual')
    ax.set_ylabel('Z Error vs GT (mm)')
    ax.set_title(f'Local: Z Error vs Residual\n({len(local_errors_z_matched)} landmarks)')
    ax.grid(True, alpha=0.3)
    if len(local_residuals) > 1:
        corr = np.corrcoef(local_residuals, local_errors_z_matched)[0, 1]
        ax.text(0.05, 0.95, f'Corr: {corr:.3f}', 
               transform=ax.transAxes, fontsize=9, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Global optimizer - X error
ax = axes[1, 0]
if len(global_residuals) > 0 and len(global_errors_x_matched) > 0:
    ax.scatter(global_residuals, global_errors_x_matched * 1000, 
              alpha=0.6, s=30, c='magenta', edgecolors='darkmagenta', linewidths=0.5)
    ax.set_xlabel('Factor Residual')
    ax.set_ylabel('X Error vs GT (mm)')
    ax.set_title(f'Global: X Error vs Residual\n({len(global_errors_x_matched)} landmarks)')
    ax.grid(True, alpha=0.3)
    if len(global_residuals) > 1:
        corr = np.corrcoef(global_residuals, global_errors_x_matched)[0, 1]
        ax.text(0.05, 0.95, f'Corr: {corr:.3f}', 
               transform=ax.transAxes, fontsize=9, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Global optimizer - Y error
ax = axes[1, 1]
if len(global_residuals) > 0 and len(global_errors_y_matched) > 0:
    ax.scatter(global_residuals, global_errors_y_matched * 1000, 
              alpha=0.6, s=30, c='magenta', edgecolors='darkmagenta', linewidths=0.5)
    ax.set_xlabel('Factor Residual')
    ax.set_ylabel('Y Error vs GT (mm)')
    ax.set_title(f'Global: Y Error vs Residual\n({len(global_errors_y_matched)} landmarks)')
    ax.grid(True, alpha=0.3)
    if len(global_residuals) > 1:
        corr = np.corrcoef(global_residuals, global_errors_y_matched)[0, 1]
        ax.text(0.05, 0.95, f'Corr: {corr:.3f}', 
               transform=ax.transAxes, fontsize=9, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Global optimizer - Z error
ax = axes[1, 2]
if len(global_residuals) > 0 and len(global_errors_z_matched) > 0:
    ax.scatter(global_residuals, global_errors_z_matched * 1000, 
              alpha=0.6, s=30, c='magenta', edgecolors='darkmagenta', linewidths=0.5)
    ax.set_xlabel('Factor Residual')
    ax.set_ylabel('Z Error vs GT (mm)')
    ax.set_title(f'Global: Z Error vs Residual\n({len(global_errors_z_matched)} landmarks)')
    ax.grid(True, alpha=0.3)
    if len(global_residuals) > 1:
        corr = np.corrcoef(global_residuals, global_errors_z_matched)[0, 1]
        ax.text(0.05, 0.95, f'Corr: {corr:.3f}', 
               transform=ax.transAxes, fontsize=9, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

# Plot 2: Landmark Position Error vs Frontend Residual (separated by X, Y, Z)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Local - X error vs frontend residual
ax = axes[0, 0]
if len(local_frontend_residuals) > 0 and len(local_errors_matched_fe) > 0:
    # Get X component errors
    local_errors_x_fe = []
    for lid in ids_local:
        if lid in frontend_residuals_local_dict and not np.isnan(frontend_residuals_local_dict[lid]):
            idx = np.where(ids_local == lid)[0][0]
            local_errors_x_fe.append(local_errors_x[idx])
    local_errors_x_fe = np.array(local_errors_x_fe)
    
    if len(local_errors_x_fe) > 0:
        ax.scatter(local_frontend_residuals, local_errors_x_fe * 1000, 
                  alpha=0.6, s=30, c='blue', edgecolors='darkblue', linewidths=0.5)
        ax.set_xlabel('Frontend Residual')
        ax.set_ylabel('X Error vs GT (mm)')
        ax.set_title(f'Local: X Error vs Frontend Residual\n({len(local_errors_x_fe)} landmarks)')
        ax.grid(True, alpha=0.3)
        if len(local_frontend_residuals) > 1:
            corr = np.corrcoef(local_frontend_residuals, local_errors_x_fe)[0, 1]
            ax.text(0.05, 0.95, f'Corr: {corr:.3f}', 
                   transform=ax.transAxes, fontsize=9, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Local - Y error vs frontend residual
ax = axes[0, 1]
if len(local_frontend_residuals) > 0:
    local_errors_y_fe = []
    for lid in ids_local:
        if lid in frontend_residuals_local_dict and not np.isnan(frontend_residuals_local_dict[lid]):
            idx = np.where(ids_local == lid)[0][0]
            local_errors_y_fe.append(local_errors_y[idx])
    local_errors_y_fe = np.array(local_errors_y_fe)
    
    if len(local_errors_y_fe) > 0:
        ax.scatter(local_frontend_residuals, local_errors_y_fe * 1000, 
                  alpha=0.6, s=30, c='blue', edgecolors='darkblue', linewidths=0.5)
        ax.set_xlabel('Frontend Residual')
        ax.set_ylabel('Y Error vs GT (mm)')
        ax.set_title(f'Local: Y Error vs Frontend Residual\n({len(local_errors_y_fe)} landmarks)')
        ax.grid(True, alpha=0.3)
        if len(local_frontend_residuals) > 1:
            corr = np.corrcoef(local_frontend_residuals, local_errors_y_fe)[0, 1]
            ax.text(0.05, 0.95, f'Corr: {corr:.3f}', 
                   transform=ax.transAxes, fontsize=9, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Local - Z error vs frontend residual
ax = axes[0, 2]
if len(local_frontend_residuals) > 0:
    local_errors_z_fe = []
    for lid in ids_local:
        if lid in frontend_residuals_local_dict and not np.isnan(frontend_residuals_local_dict[lid]):
            idx = np.where(ids_local == lid)[0][0]
            local_errors_z_fe.append(local_errors_z[idx])
    local_errors_z_fe = np.array(local_errors_z_fe)
    
    if len(local_errors_z_fe) > 0:
        ax.scatter(local_frontend_residuals, local_errors_z_fe * 1000, 
                  alpha=0.6, s=30, c='blue', edgecolors='darkblue', linewidths=0.5)
        ax.set_xlabel('Frontend Residual')
        ax.set_ylabel('Z Error vs GT (mm)')
        ax.set_title(f'Local: Z Error vs Frontend Residual\n({len(local_errors_z_fe)} landmarks)')
        ax.grid(True, alpha=0.3)
        if len(local_frontend_residuals) > 1:
            corr = np.corrcoef(local_frontend_residuals, local_errors_z_fe)[0, 1]
            ax.text(0.05, 0.95, f'Corr: {corr:.3f}', 
                   transform=ax.transAxes, fontsize=9, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Global - X error vs frontend residual
ax = axes[1, 0]
if len(global_frontend_residuals) > 0 and len(global_errors_matched_fe) > 0:
    global_errors_x_fe = []
    for lid in ids_global:
        if lid in frontend_residuals_global_dict and not np.isnan(frontend_residuals_global_dict[lid]):
            idx = np.where(ids_global == lid)[0][0]
            global_errors_x_fe.append(global_errors_x[idx])
    global_errors_x_fe = np.array(global_errors_x_fe)
    
    if len(global_errors_x_fe) > 0:
        ax.scatter(global_frontend_residuals, global_errors_x_fe * 1000, 
                  alpha=0.6, s=30, c='magenta', edgecolors='darkmagenta', linewidths=0.5)
        ax.set_xlabel('Frontend Residual')
        ax.set_ylabel('X Error vs GT (mm)')
        ax.set_title(f'Global: X Error vs Frontend Residual\n({len(global_errors_x_fe)} landmarks)')
        ax.grid(True, alpha=0.3)
        if len(global_frontend_residuals) > 1:
            corr = np.corrcoef(global_frontend_residuals, global_errors_x_fe)[0, 1]
            ax.text(0.05, 0.95, f'Corr: {corr:.3f}', 
                   transform=ax.transAxes, fontsize=9, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Global - Y error vs frontend residual
ax = axes[1, 1]
if len(global_frontend_residuals) > 0:
    global_errors_y_fe = []
    for lid in ids_global:
        if lid in frontend_residuals_global_dict and not np.isnan(frontend_residuals_global_dict[lid]):
            idx = np.where(ids_global == lid)[0][0]
            global_errors_y_fe.append(global_errors_y[idx])
    global_errors_y_fe = np.array(global_errors_y_fe)
    
    if len(global_errors_y_fe) > 0:
        ax.scatter(global_frontend_residuals, global_errors_y_fe * 1000, 
                  alpha=0.6, s=30, c='magenta', edgecolors='darkmagenta', linewidths=0.5)
        ax.set_xlabel('Frontend Residual')
        ax.set_ylabel('Y Error vs GT (mm)')
        ax.set_title(f'Global: Y Error vs Frontend Residual\n({len(global_errors_y_fe)} landmarks)')
        ax.grid(True, alpha=0.3)
        if len(global_frontend_residuals) > 1:
            corr = np.corrcoef(global_frontend_residuals, global_errors_y_fe)[0, 1]
            ax.text(0.05, 0.95, f'Corr: {corr:.3f}', 
                   transform=ax.transAxes, fontsize=9, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Global - Z error vs frontend residual
ax = axes[1, 2]
if len(global_frontend_residuals) > 0:
    global_errors_z_fe = []
    for lid in ids_global:
        if lid in frontend_residuals_global_dict and not np.isnan(frontend_residuals_global_dict[lid]):
            idx = np.where(ids_global == lid)[0][0]
            global_errors_z_fe.append(global_errors_z[idx])
    global_errors_z_fe = np.array(global_errors_z_fe)
    
    if len(global_errors_z_fe) > 0:
        ax.scatter(global_frontend_residuals, global_errors_z_fe * 1000, 
                  alpha=0.6, s=30, c='magenta', edgecolors='darkmagenta', linewidths=0.5)
        ax.set_xlabel('Frontend Residual')
        ax.set_ylabel('Z Error vs GT (mm)')
        ax.set_title(f'Global: Z Error vs Frontend Residual\n({len(global_errors_z_fe)} landmarks)')
        ax.grid(True, alpha=0.3)
        if len(global_frontend_residuals) > 1:
            corr = np.corrcoef(global_frontend_residuals, global_errors_z_fe)[0, 1]
            ax.text(0.05, 0.95, f'Corr: {corr:.3f}', 
                   transform=ax.transAxes, fontsize=9, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

# Plot 3: Landmark 3D Norm Error vs Frontend Residual
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Local - 3D norm error vs frontend residual
ax = axes[0]
if len(local_frontend_residuals) > 0 and len(local_errors_matched_fe) > 0:
    ax.scatter(local_frontend_residuals, local_errors_matched_fe * 1000, 
              alpha=0.6, s=30, c='blue', edgecolors='darkblue', linewidths=0.5)
    ax.set_xlabel('Frontend Residual')
    ax.set_ylabel('3D Position Error vs GT (mm)')
    ax.set_title(f'Local: 3D Norm Error vs Frontend Residual\n({len(local_errors_matched_fe)} landmarks)')
    ax.grid(True, alpha=0.3)
    if len(local_frontend_residuals) > 1:
        corr = np.corrcoef(local_frontend_residuals, local_errors_matched_fe)[0, 1]
        ax.text(0.05, 0.95, f'Correlation: {corr:.3f}', 
               transform=ax.transAxes, fontsize=10, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
else:
    ax.text(0.5, 0.5, 'No data available', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Local: 3D Norm Error vs Frontend Residual')

# Global - 3D norm error vs frontend residual
ax = axes[1]
if len(global_frontend_residuals) > 0 and len(global_errors_matched_fe) > 0:
    ax.scatter(global_frontend_residuals, global_errors_matched_fe * 1000, 
              alpha=0.6, s=30, c='magenta', edgecolors='darkmagenta', linewidths=0.5)
    ax.set_xlabel('Frontend Residual')
    ax.set_ylabel('3D Position Error vs GT (mm)')
    ax.set_title(f'Global: 3D Norm Error vs Frontend Residual\n({len(global_errors_matched_fe)} landmarks)')
    ax.grid(True, alpha=0.3)
    if len(global_frontend_residuals) > 1:
        corr = np.corrcoef(global_frontend_residuals, global_errors_matched_fe)[0, 1]
        ax.text(0.05, 0.95, f'Correlation: {corr:.3f}', 
               transform=ax.transAxes, fontsize=10, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
else:
    ax.text(0.5, 0.5, 'No data available', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Global: 3D Norm Error vs Frontend Residual')

plt.tight_layout()
plt.show()


In [ ]:
# Error Analysis: Detailed error plots (similar to notebook 12)

# Compute errors
local_errors_3d = np.linalg.norm(gt_local - est_local, axis=1) if len(gt_local) > 0 else np.array([])
global_errors_3d = np.linalg.norm(gt_global - est_global, axis=1) if len(gt_global) > 0 else np.array([])

local_errors_x = (gt_local[:, 0] - est_local[:, 0]) if len(gt_local) > 0 else np.array([])
local_errors_y = (gt_local[:, 1] - est_local[:, 1]) if len(gt_local) > 0 else np.array([])
local_errors_z = (gt_local[:, 2] - est_local[:, 2]) if len(gt_local) > 0 else np.array([])

global_errors_x = (gt_global[:, 0] - est_global[:, 0]) if len(gt_global) > 0 else np.array([])
global_errors_y = (gt_global[:, 1] - est_global[:, 1]) if len(gt_global) > 0 else np.array([])
global_errors_z = (gt_global[:, 2] - est_global[:, 2]) if len(gt_global) > 0 else np.array([])

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Landmark errors per component (Local)
ax = axes[0, 0]
if len(local_errors_x) > 0:
    ax.scatter(range(len(local_errors_x)), local_errors_x * 1000, 
              label='X error', alpha=0.6, s=20, c='r')
    ax.scatter(range(len(local_errors_y)), local_errors_y * 1000, 
              label='Y error', alpha=0.6, s=20, c='g')
    ax.scatter(range(len(local_errors_z)), local_errors_z * 1000, 
              label='Z error', alpha=0.6, s=20, c='b')
    ax.set_xlabel('Landmark Index')
    ax.set_ylabel('Position Error (mm)')
    ax.set_title('Local: Landmark Position Errors (per component)')
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No landmarks available', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Local: Landmark Position Errors')

# Landmark errors per component (Global)
ax = axes[0, 1]
if len(global_errors_x) > 0:
    ax.scatter(range(len(global_errors_x)), global_errors_x * 1000, 
              label='X error', alpha=0.6, s=20, c='r')
    ax.scatter(range(len(global_errors_y)), global_errors_y * 1000, 
              label='Y error', alpha=0.6, s=20, c='g')
    ax.scatter(range(len(global_errors_z)), global_errors_z * 1000, 
              label='Z error', alpha=0.6, s=20, c='b')
    ax.set_xlabel('Landmark Index')
    ax.set_ylabel('Position Error (mm)')
    ax.set_title('Global: Landmark Position Errors (per component)')
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No landmarks available', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Global: Landmark Position Errors')

# Comparison: Local vs Global error magnitude
ax = axes[0, 2]
if len(local_errors_3d) > 0 and len(global_errors_3d) > 0:
    ax.scatter(range(len(local_errors_3d)), local_errors_3d * 1000, 
              label='Local', alpha=0.6, s=20, c='blue')
    ax.scatter(range(len(global_errors_3d)), global_errors_3d * 1000, 
              label='Global', alpha=0.6, s=20, c='magenta')
    ax.set_xlabel('Landmark Index')
    ax.set_ylabel('3D Position Error (mm)')
    ax.set_title('3D Error Comparison: Local vs Global')
    ax.legend()
    ax.grid(True, alpha=0.3)
elif len(local_errors_3d) > 0:
    ax.scatter(range(len(local_errors_3d)), local_errors_3d * 1000, 
              label='Local', alpha=0.6, s=20, c='blue')
    ax.set_xlabel('Landmark Index')
    ax.set_ylabel('3D Position Error (mm)')
    ax.set_title('3D Error Comparison: Local vs Global')
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No landmarks available', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('3D Error Comparison')

# Error distribution (Local)
ax = axes[1, 0]
if len(local_errors_3d) > 0:
    ax.hist(local_errors_3d * 1000, bins=30, alpha=0.7, edgecolor='black', color='blue')
    ax.axvline(np.mean(local_errors_3d) * 1000, color='red', linestyle='--', linewidth=2, 
              label=f'Mean: {np.mean(local_errors_3d)*1000:.2f} mm')
    ax.axvline(np.median(local_errors_3d) * 1000, color='green', linestyle='--', linewidth=2, 
              label=f'Median: {np.median(local_errors_3d)*1000:.2f} mm')
    ax.set_xlabel('3D Position Error (mm)')
    ax.set_ylabel('Frequency')
    ax.set_title('Local: Error Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
else:
    ax.text(0.5, 0.5, 'No landmarks available', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Local: Error Distribution')

# Error distribution (Global)
ax = axes[1, 1]
if len(global_errors_3d) > 0:
    ax.hist(global_errors_3d * 1000, bins=30, alpha=0.7, edgecolor='black', color='magenta')
    ax.axvline(np.mean(global_errors_3d) * 1000, color='red', linestyle='--', linewidth=2, 
              label=f'Mean: {np.mean(global_errors_3d)*1000:.2f} mm')
    ax.axvline(np.median(global_errors_3d) * 1000, color='green', linestyle='--', linewidth=2, 
              label=f'Median: {np.median(global_errors_3d)*1000:.2f} mm')
    ax.set_xlabel('3D Position Error (mm)')
    ax.set_ylabel('Frequency')
    ax.set_title('Global: Error Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
else:
    ax.text(0.5, 0.5, 'No landmarks available', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Global: Error Distribution')

# Statistics panel
ax = axes[1, 2]
ax.axis('off')

stats_text = "Landmark Error Statistics\n"
stats_text += "=" * 50 + "\n\n"

if len(local_errors_3d) > 0:
    stats_text += "Local Optimization:\n"
    stats_text += f"  Mean Error: {np.mean(local_errors_3d)*1000:.2f} mm\n"
    stats_text += f"  Median Error: {np.median(local_errors_3d)*1000:.2f} mm\n"
    stats_text += f"  Std Dev: {np.std(local_errors_3d)*1000:.2f} mm\n"
    stats_text += f"  Max Error: {np.max(local_errors_3d)*1000:.2f} mm\n"
    stats_text += f"  Min Error: {np.min(local_errors_3d)*1000:.2f} mm\n"
    stats_text += f"  # Landmarks: {len(local_errors_3d)}\n\n"

if len(global_errors_3d) > 0:
    stats_text += "Global Optimization:\n"
    stats_text += f"  Mean Error: {np.mean(global_errors_3d)*1000:.2f} mm\n"
    stats_text += f"  Median Error: {np.median(global_errors_3d)*1000:.2f} mm\n"
    stats_text += f"  Std Dev: {np.std(global_errors_3d)*1000:.2f} mm\n"
    stats_text += f"  Max Error: {np.max(global_errors_3d)*1000:.2f} mm\n"
    stats_text += f"  Min Error: {np.min(global_errors_3d)*1000:.2f} mm\n"
    stats_text += f"  # Landmarks: {len(global_errors_3d)}\n"

if len(local_errors_3d) > 0 and len(global_errors_3d) > 0:
    improvement = (1 - np.mean(global_errors_3d) / np.mean(local_errors_3d)) * 100
    stats_text += f"\nGlobal vs Local Improvement: {improvement:.1f}%\n"

ax.text(0.05, 0.5, stats_text, fontsize=10, family='monospace', 
        verticalalignment='center', fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Side-by-side comparison: Local vs Global factor graphs
# Shows both the local (all frames) and global (keyframes only) optimization graphs

fig = plt.figure(figsize=(16, 6))

# Local factor graph (left)
ax1 = fig.add_subplot(121, projection='3d')
plot_factor_graph_local(ax1, local_optimizer, title="Local Factor Graph (All Frames)")

# Global keyframe factor graph (right)
ax2 = fig.add_subplot(122, projection='3d')
if len(keyframe_indices) > 0:
    plot_factor_graph_global(ax2, kf_graph, keyframe_indices, traj_global, 
                             title="Global Factor Graph (Keyframes Only)")
else:
    ax2.text(0.5, 0.5, 'No keyframes available', ha='center', va='center', transform=ax2.transAxes)
    ax2.set_title("Global Factor Graph (Keyframes Only)")

plt.tight_layout()
plt.show()


In [ ]:
sum(global_errors_3d>0.010)

np.arange(len(global_errors_3d))[global_errors_3d>0.010]